# EMA Crossover Scanner — SL & Target as Input Arguments

Same EMA cross + support-touch strategy as `ema_strategy_full.ipynb`, but **SL %** and **Target %** are taken as input arguments.

- `SL_PCT`     — Stop Loss as % below Touch Close (e.g. `1.5` → SL = Touch Close × 0.985)
- `TARGET_PCT` — Target as % above Touch Close (e.g. `3.0` → Target = Touch Close × 1.03)

Run the **Inputs** cell to be prompted interactively, or just edit the values directly.

In [135]:
# ============================================================
#  INPUTS — SL % and TARGET %
# ============================================================
# Set USE_PROMPT = True to type values interactively when running the cell.
# Set USE_PROMPT = False to use the hard-coded defaults below.

USE_PROMPT = False

SL_PCT     = 0.7     # default stop-loss % below Touch Close
TARGET_PCT = 3.7     # default target %     above Touch Close

if USE_PROMPT:
    try:
        SL_PCT     = float(input('Enter Stop Loss % (e.g. 1.5): ').strip() or SL_PCT)
        TARGET_PCT = float(input('Enter Target % (e.g. 3.0): ').strip() or TARGET_PCT)
    except Exception as e:
        print(f'Invalid input, using defaults. ({e})')

print('=' * 50)
print(f'  SL_PCT     = {SL_PCT}%   (SL = Touch Close × {1 - SL_PCT/100:.4f})')
print(f'  TARGET_PCT = {TARGET_PCT}%   (TP = Touch Close × {1 + TARGET_PCT/100:.4f})')
print(f'  Risk:Reward = 1 : {TARGET_PCT/SL_PCT:.2f}')
print('=' * 50)

  SL_PCT     = 0.7%   (SL = Touch Close × 0.9930)
  TARGET_PCT = 3.7%   (TP = Touch Close × 1.0370)
  Risk:Reward = 1 : 5.29


In [136]:
# ============================================================
#  CONFIG  —  Stocks / Files / Setups
# ============================================================

STOCKS = [
    'RELIANCE',
    'TCS',
    'HDFCBANK',
    'INFY',
    'ICICIBANK',
]

FILE_MAP = {
    '1d': 'data/1d/{symbol}_historical.csv',
    '1h': 'data/1h/{symbol}_historical.csv',
}

SETUPS = [
    ['1d',  21, 50],
    ['1d',  10, 20],
    ['1h',  21, 50],
    ['1h',  10, 20],
]

DATA_FOLDER = '.'

In [137]:
import pandas as pd
import numpy as np
import os
from IPython.display import display

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows',    300)
pd.set_option('display.width',       300)
pd.set_option('display.float_format', '{:.2f}'.format)

missing = [s[0] for s in SETUPS if s[0] not in FILE_MAP]
if missing:
    raise ValueError(f'Interval(s) {missing} not in FILE_MAP.')

TIMEFRAMES = list(dict.fromkeys([s[0] for s in SETUPS]))
print(f'Stocks: {STOCKS}')
print(f'Setups: {SETUPS}')

Stocks: ['RELIANCE', 'TCS', 'HDFCBANK', 'INFY', 'ICICIBANK']
Setups: [['1d', 21, 50], ['1d', 10, 20], ['1h', 21, 50], ['1h', 10, 20]]


In [138]:
# ── LOAD CSV ───────────────────────────────────────────────────────────
def load_csv(symbol, interval):
    filename = FILE_MAP[interval].replace('{symbol}', symbol)
    filepath = os.path.join(DATA_FOLDER, filename)
    if not os.path.exists(filepath):
        raise FileNotFoundError(f'Not found: {filepath}')
    df = pd.read_csv(filepath)
    df.columns = [c.strip().lower() for c in df.columns]
    if 'datetime' in df.columns:
        df['Date'] = pd.to_datetime(df['datetime'])
    elif 'timestamp' in df.columns:
        df['Date'] = pd.to_datetime(df['timestamp'], unit='s') + pd.Timedelta(hours=5, minutes=30)
    else:
        raise ValueError(f'No datetime/timestamp column in {filename}')
    df = df.set_index('Date').sort_index()
    if df.index.tz is not None:
        df.index = df.index.tz_localize(None)
    df = df.rename(columns={'open':'Open','high':'High','low':'Low','close':'Close','volume':'Volume'})
    cols = [c for c in ['Open','High','Low','Close','Volume'] if c in df.columns]
    return df[cols].dropna()

print('load_csv() ready.')

load_csv() ready.


In [139]:
# ── EMA + SCAN ─────────────────────────────────────────────────────────
def add_emas(df, fast, slow):
    df = df.copy()
    df[f'EMA{fast}'] = df['Close'].ewm(span=fast, adjust=False).mean()
    df[f'EMA{slow}'] = df['Close'].ewm(span=slow, adjust=False).mean()
    return df

def bullish_cross_indices(df, fast, slow):
    above = df[f'EMA{fast}'] > df[f'EMA{slow}']
    cross = above & ~above.shift(1, fill_value=False)
    cross.iloc[0] = False
    return list(np.where(cross.values)[0])

def touch_zone(low, ef, es):
    return 'At/Below Slow EMA' if low <= es else 'Between Slow & Fast EMA'

def scan(stock, interval, fast, slow, df):
    df = add_emas(df, fast, slow)
    rows = []
    for cidx in bullish_cross_indices(df, fast, slow):
        cross_ts = df.index[cidx]
        for i in range(cidx + 1, len(df)):
            low, close, open_, high = (float(df['Low'].iloc[i]),
                                       float(df['Close'].iloc[i]),
                                       float(df['Open'].iloc[i]),
                                       float(df['High'].iloc[i]))
            ef = float(df[f'EMA{fast}'].iloc[i]); es = float(df[f'EMA{slow}'].iloc[i])
            if ef <= es:
                break
            if low <= ef and close > ef and close > open_:
                rows.append({
                    'Stock': stock, 'Timeframe': interval,
                    'EMA Pair': f'EMA{fast}/EMA{slow}',
                    'Cross Time': str(cross_ts),
                    'Touch Time': str(df.index[i]),
                    'Touch Open': round(open_, 2), 'Touch High': round(high, 2),
                    'Touch Low':  round(low,  2), 'Touch Close': round(close, 2),
                    f'EMA{fast} @ Touch': round(ef, 2),
                    f'EMA{slow} @ Touch': round(es, 2),
                    'Touch Zone': touch_zone(low, ef, es),
                    'Candles After Cross': i - cidx,
                })
    return rows

print('scan() ready.')

scan() ready.


In [140]:
# ── RUN SCANNER ────────────────────────────────────────────────────────
all_signals = []
stock_data  = {}

for setup in SETUPS:
    interval, fast, slow = setup
    for stock in STOCKS:
        try:
            df = load_csv(stock, interval)
            stock_data[(stock, interval)] = add_emas(df, fast, slow)
            sig = scan(stock, interval, fast, slow, df)
            all_signals.extend(sig)
            print(f'  {stock:10s} [{interval}] EMA{fast}/EMA{slow} → {len(sig)} signals')
        except Exception as e:
            print(f'  {stock} [{interval}] ERROR: {e}')

signals = pd.DataFrame(all_signals)
if not signals.empty:
    signals = signals.sort_values(['Stock','Timeframe','EMA Pair','Cross Time','Touch Time']).reset_index(drop=True)
    signals.index += 1

print(f'\nTotal signals: {len(signals)}')

  RELIANCE   [1d] EMA21/EMA50 → 24 signals
  TCS        [1d] EMA21/EMA50 → 19 signals
  HDFCBANK   [1d] EMA21/EMA50 → 40 signals
  INFY       [1d] EMA21/EMA50 → 22 signals
  ICICIBANK  [1d] EMA21/EMA50 → 36 signals
  RELIANCE   [1d] EMA10/EMA20 → 29 signals
  TCS        [1d] EMA10/EMA20 → 35 signals
  HDFCBANK   [1d] EMA10/EMA20 → 56 signals
  INFY       [1d] EMA10/EMA20 → 42 signals
  ICICIBANK  [1d] EMA10/EMA20 → 67 signals
  RELIANCE   [1h] EMA21/EMA50 → 17 signals
  TCS        [1h] EMA21/EMA50 → 9 signals
  HDFCBANK   [1h] EMA21/EMA50 → 5 signals
  INFY       [1h] EMA21/EMA50 → 17 signals
  ICICIBANK  [1h] EMA21/EMA50 → 13 signals
  RELIANCE   [1h] EMA10/EMA20 → 31 signals
  TCS        [1h] EMA10/EMA20 → 24 signals
  HDFCBANK   [1h] EMA10/EMA20 → 19 signals
  INFY       [1h] EMA10/EMA20 → 26 signals
  ICICIBANK  [1h] EMA10/EMA20 → 28 signals

Total signals: 559


In [141]:
# ── APPLY SL_PCT + TARGET_PCT TO EACH SIGNAL ───────────────────────────
def candles_to_duration(n, interval):
    if interval in ('60', '1h'): return f'{n}h'
    if interval == '1d':         return f'{n}d'
    return f'{n} candles'

def evaluate(signals, stock_data, sl_pct, target_pct):
    out = []
    for _, row in signals.iterrows():
        stock    = row['Stock']
        interval = row['Timeframe']
        entry    = float(row['Touch Close'])
        touch_ts = pd.Timestamp(row['Touch Time'])

        sl_price     = round(entry * (1 - sl_pct/100),     2)
        target_price = round(entry * (1 + target_pct/100), 2)

        df = stock_data.get((stock, interval))
        if df is None:
            continue
        try:
            idx = df.index.get_loc(touch_ts)
        except KeyError:
            idx = df.index.searchsorted(touch_ts)

        sl_hit = target_hit = False
        sl_time = target_time = None
        sl_idx  = target_idx  = None
        max_high = float('-inf')

        for j in range(idx + 1, len(df)):
            hi = float(df['High'].iloc[j]); lo = float(df['Low'].iloc[j])
            max_high = max(max_high, hi)

            t_now = hi >= target_price
            s_now = lo <= sl_price

            if t_now and not target_hit:
                target_hit = True; target_time = df.index[j]; target_idx = j
            if s_now and not sl_hit:
                sl_hit = True;     sl_time     = df.index[j]; sl_idx     = j

            if target_hit or sl_hit:
                break

        # Outcome label
        if target_hit and sl_hit:
            if target_idx < sl_idx:    outcome = 'TARGET HIT'
            elif sl_idx < target_idx:  outcome = 'SL HIT'
            else:                      outcome = 'AMBIGUOUS (same candle)'
        elif target_hit:               outcome = 'TARGET HIT'
        elif sl_hit:                   outcome = 'SL HIT'
        else:                          outcome = 'OPEN'

        exit_idx = (target_idx if outcome == 'TARGET HIT'
                    else sl_idx if outcome == 'SL HIT'
                    else len(df) - 1)
        n_candles = exit_idx - idx if exit_idx is not None else None
        duration  = candles_to_duration(n_candles, interval) if n_candles is not None else None

        if outcome == 'TARGET HIT':  pnl_pct =  target_pct
        elif outcome == 'SL HIT':    pnl_pct = -sl_pct
        else:                        pnl_pct = round((float(df['Close'].iloc[-1]) - entry) / entry * 100, 2)

        out.append({
            **row,
            'Entry':         round(entry, 2),
            'SL Price':      sl_price,
            'Target Price':  target_price,
            'Outcome':       outcome,
            'SL Hit Time':     str(sl_time)     if sl_time     else 'Not Hit',
            'Target Hit Time': str(target_time) if target_time else 'Not Hit',
            'Max High Seen': round(max_high, 2) if max_high != float('-inf') else None,
            'PnL %':         pnl_pct,
            'Duration':      duration,
        })
    return pd.DataFrame(out)

results = evaluate(signals, stock_data, SL_PCT, TARGET_PCT)
print(f'Evaluated {len(results)} signals with SL={SL_PCT}%  Target={TARGET_PCT}%')

Evaluated 559 signals with SL=0.7%  Target=3.7%


In [142]:
# ── SUMMARY REPORTS ────────────────────────────────────────────────────
sep = '=' * 70
total = len(results)
tgt   = (results['Outcome'] == 'TARGET HIT').sum()
sl    = (results['Outcome'] == 'SL HIT').sum()
open_ = (results['Outcome'] == 'OPEN').sum()
amb   = (results['Outcome'] == 'AMBIGUOUS (same candle)').sum()

print(sep)
print(f'  OVERALL — SL: {SL_PCT}%   Target: {TARGET_PCT}%   (R:R = 1:{TARGET_PCT/SL_PCT:.2f})')
print(sep)
print(f'  Total signals : {total}')
print(f'  TARGET HIT    : {tgt}  ({tgt/total*100:.1f}%)')
print(f'  SL HIT        : {sl}   ({sl/total*100:.1f}%)')
print(f'  OPEN          : {open_} ({open_/total*100:.1f}%)')
if amb:
    print(f'  Ambiguous     : {amb}')
decided = tgt + sl
if decided:
    print(f'  Win rate (decided): {tgt/decided*100:.1f}%')
expectancy = results['PnL %'].mean()
print(f'  Avg PnL %     : {expectancy:.2f}%')
print(sep)

# Per setup
print('\n  Per Setup:')
per_setup = (results.groupby(['Timeframe', 'EMA Pair'])
             .apply(lambda x: pd.Series({
                 'Total':       len(x),
                 'Target Hit':  (x['Outcome'] == 'TARGET HIT').sum(),
                 'SL Hit':      (x['Outcome'] == 'SL HIT').sum(),
                 'Open':        (x['Outcome'] == 'OPEN').sum(),
                 'Win Rate %':  round((x['Outcome'] == 'TARGET HIT').mean() * 100, 1),
                 'Avg PnL %':   round(x['PnL %'].mean(), 2),
             })).reset_index())
display(per_setup)

# Per stock
print('\n  Per Stock:')
per_stock = (results.groupby('Stock')
             .apply(lambda x: pd.Series({
                 'Total':       len(x),
                 'Target Hit':  (x['Outcome'] == 'TARGET HIT').sum(),
                 'SL Hit':      (x['Outcome'] == 'SL HIT').sum(),
                 'Win Rate %':  round((x['Outcome'] == 'TARGET HIT').mean() * 100, 1),
                 'Avg PnL %':   round(x['PnL %'].mean(), 2),
             })).reset_index())
display(per_stock)

  OVERALL — SL: 0.7%   Target: 3.7%   (R:R = 1:5.29)
  Total signals : 559
  TARGET HIT    : 83  (14.8%)
  SL HIT        : 474   (84.8%)
  OPEN          : 2 (0.4%)
  Win rate (decided): 14.9%
  Avg PnL %     : -0.04%

  Per Setup:


,Timeframe,EMA Pair,Total,Target Hit,SL Hit,Open,Win Rate %,Avg PnL %
0,1d,EMA10/EMA20,229.00,33.00,195.00,1.00,14.40,-0.06
1,1d,EMA21/EMA50,141.00,28.00,113.00,0.00,19.90,0.17
2,1h,EMA10/EMA20,128.00,13.00,115.00,0.00,10.20,-0.25
3,1h,EMA21/EMA50,61.00,9.00,51.00,1.00,14.80,-0.04



  Per Stock:


,Stock,Total,Target Hit,SL Hit,Win Rate %,Avg PnL %
0,HDFCBANK,120.00,15.00,105.00,12.50,-0.15
1,ICICIBANK,144.00,20.00,124.00,13.90,-0.09
2,INFY,107.00,16.00,91.00,15.00,-0.04
3,RELIANCE,101.00,16.00,83.00,15.80,0.01
4,TCS,87.00,16.00,71.00,18.40,0.11


In [143]:
# ── FULL RESULTS TABLE ─────────────────────────────────────────────────
display(results[['Stock', 'Timeframe', 'EMA Pair', 'Touch Time',
                 'Entry', 'SL Price', 'Target Price', 'Outcome',
                 'SL Hit Time', 'Target Hit Time',
                 'Max High Seen', 'PnL %', 'Duration']])

,Stock,Timeframe,EMA Pair,Touch Time,Entry,SL Price,Target Price,Outcome,SL Hit Time,Target Hit Time,Max High Seen,PnL %,Duration
0,HDFCBANK,1d,EMA10/EMA20,2024-03-15 00:00:00,726.33,721.25,753.20,SL HIT,2024-03-18 00:00:00,Not Hit,728.00,-0.70,1d
1,HDFCBANK,1d,EMA10/EMA20,2024-03-19 00:00:00,724.67,719.60,751.48,SL HIT,2024-03-20 00:00:00,Not Hit,725.83,-0.70,1d
2,HDFCBANK,1d,EMA10/EMA20,2024-03-21 00:00:00,722.88,717.82,749.63,SL HIT,2024-03-26 00:00:00,Not Hit,725.38,-0.70,2d
3,HDFCBANK,1d,EMA10/EMA20,2024-04-19 00:00:00,765.65,760.29,793.98,SL HIT,2024-04-22 00:00:00,Not Hit,778.70,-0.70,1d
4,HDFCBANK,1d,EMA10/EMA20,2024-04-29 00:00:00,764.75,759.40,793.05,SL HIT,2024-04-30 00:00:00,Not Hit,769.75,-0.70,1d
...,...,...,...,...,...,...,...,...,...,...,...,...,...
554,TCS,1h,EMA21/EMA50,2026-04-02 11:15:00,2406.60,2389.75,2495.64,TARGET HIT,Not Hit,2026-04-07 09:15:00,2500.00,3.70,12h
555,TCS,1h,EMA21/EMA50,2026-04-02 12:15:00,2419.80,2402.86,2509.33,TARGET HIT,Not Hit,2026-04-07 10:15:00,2527.10,3.70,12h
556,TCS,1h,EMA21/EMA50,2026-04-07 09:15:00,2499.80,2482.30,2592.29,TARGET HIT,Not Hit,2026-04-09 11:15:00,2598.80,3.70,16h
557,TCS,1h,EMA21/EMA50,2026-04-15 09:15:00,2549.30,2531.45,2643.62,SL HIT,2026-04-15 10:15:00,Not Hit,2554.70,-0.70,1h


In [144]:
# ── EXPORT ─────────────────────────────────────────────────────────────
signals_dir = 'signals'
os.makedirs(signals_dir, exist_ok=True)

tag = f'sl{SL_PCT}_tp{TARGET_PCT}'.replace('.', 'p')
out = os.path.join(signals_dir, f'signals_{tag}.csv')
results.to_csv(out, index=False)
print(f'Saved → {out}')

for (tf, pair), grp in results.groupby(['Timeframe', 'EMA Pair']):
    fname = f'signals_{tf}_{pair.replace("/","_")}_{tag}.csv'
    fpath = os.path.join(signals_dir, fname)
    grp.to_csv(fpath, index=False)
    print(f'Saved → {fpath}  ({len(grp)} rows)')

Saved → signals\signals_sl0p7_tp3p7.csv
Saved → signals\signals_1d_EMA10_EMA20_sl0p7_tp3p7.csv  (229 rows)
Saved → signals\signals_1d_EMA21_EMA50_sl0p7_tp3p7.csv  (141 rows)
Saved → signals\signals_1h_EMA10_EMA20_sl0p7_tp3p7.csv  (128 rows)
Saved → signals\signals_1h_EMA21_EMA50_sl0p7_tp3p7.csv  (61 rows)


In [145]:
# ── OPTIMIZE SL % & TARGET % FOR MAX PnL ───────────────────────────────
# Sweeps a grid of SL/Target combos, re-evaluates all signals, and ranks
# them by average PnL %.  Also reports win-rate and expectancy.

SL_GRID     = [1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0, 6.0, 8.0]
TARGET_GRID = [1.0, 1.5, 2.0, 3.0, 4.0, 5.0, 7.0, 10.0, 15.0]

grid_rows = []
for sl in SL_GRID:
    for tp in TARGET_GRID:
        res = evaluate(signals, stock_data, sl, tp)
        tgt_n = (res['Outcome'] == 'TARGET HIT').sum()
        sl_n  = (res['Outcome'] == 'SL HIT').sum()
        decided = tgt_n + sl_n
        grid_rows.append({
            'SL %':       sl,
            'Target %':   tp,
            'R:R':        round(tp / sl, 2),
            'Total':      len(res),
            'Target Hit': int(tgt_n),
            'SL Hit':     int(sl_n),
            'Open':       int((res['Outcome'] == 'OPEN').sum()),
            'Win Rate %': round(tgt_n / decided * 100, 1) if decided else 0.0,
            'Avg PnL %':  round(res['PnL %'].mean(), 3),
            'Total PnL %':round(res['PnL %'].sum(),  2),
        })

opt = pd.DataFrame(grid_rows).sort_values('Avg PnL %', ascending=False).reset_index(drop=True)

print('=' * 70)
print('  TOP 10 SL / TARGET COMBINATIONS (ranked by Avg PnL %)')
print('=' * 70)
display(opt.head(10))

best = opt.iloc[0]
print('\n  >>> BEST COMBO <<<')
print(f"  SL = {best['SL %']}%   Target = {best['Target %']}%   R:R = 1:{best['R:R']}")
print(f"  Win Rate  : {best['Win Rate %']}%")
print(f"  Avg PnL % : {best['Avg PnL %']}%")
print(f"  Total PnL : {best['Total PnL %']}% across {best['Total']} signals")

# Save the full optimization grid
opt.to_csv(os.path.join(signals_dir, 'optimization_grid.csv'), index=False)
print(f"\nSaved full grid → {os.path.join(signals_dir, 'optimization_grid.csv')}")

# Optional: pivot for a heat-map style view
pivot = opt.pivot(index='SL %', columns='Target %', values='Avg PnL %')
print('\nAvg PnL % heat-map (rows = SL %, cols = Target %):')
display(pivot)

  TOP 10 SL / TARGET COMBINATIONS (ranked by Avg PnL %)


,SL %,Target %,R:R,Total,Target Hit,SL Hit,Open,Win Rate %,Avg PnL %,Total PnL %
0,1.00,4.00,4.00,559,105,452,2,18.90,-0.06,-32.00
1,1.00,5.00,5.00,559,87,469,3,15.60,-0.06,-32.27
2,1.00,2.00,2.00,559,160,397,2,28.70,-0.14,-77.00
3,1.00,3.00,3.00,559,119,438,2,21.40,-0.14,-81.00
4,4.00,1.50,0.38,559,382,171,6,69.10,-0.21,-116.66
5,1.00,1.50,1.50,559,195,353,2,35.60,-0.23,-125.72
6,1.50,4.00,2.67,559,128,428,3,23.00,-0.23,-130.99
7,4.00,1.00,0.25,559,418,137,4,75.30,-0.24,-132.64
8,1.50,2.00,1.33,559,199,357,3,35.80,-0.25,-138.49
9,1.00,7.00,7.00,559,51,503,5,9.20,-0.25,-139.91



  >>> BEST COMBO <<<
  SL = 1.0%   Target = 4.0%   R:R = 1:4.0
  Win Rate  : 18.9%
  Avg PnL % : -0.057%
  Total PnL : -32.0% across 559.0 signals

Saved full grid → signals\optimization_grid.csv

Avg PnL % heat-map (rows = SL %, cols = Target %):


Target %,1.00,1.50,2.00,3.00,4.00,5.00,7.00,10.00,15.00
SL %,,,,,,,,,
1.00,-0.39,-0.23,-0.14,-0.14,-0.06,-0.06,-0.25,-0.50,-0.70
1.50,-0.42,-0.32,-0.25,-0.32,-0.23,-0.28,-0.50,-0.81,-0.95
2.00,-0.43,-0.33,-0.31,-0.46,-0.42,-0.47,-0.76,-1.00,-1.09
2.50,-0.34,-0.29,-0.32,-0.44,-0.38,-0.48,-0.83,-1.08,-1.09
3.00,-0.36,-0.31,-0.35,-0.47,-0.41,-0.54,-0.96,-1.33,-1.35
4.00,-0.24,-0.21,-0.32,-0.35,-0.40,-0.62,-1.16,-1.53,-1.30
5.00,-0.32,-0.29,-0.43,-0.47,-0.60,-0.94,-1.57,-2.00,-1.73
6.00,-0.45,-0.43,-0.54,-0.73,-1.03,-1.45,-2.15,-2.45,-2.27
8.00,-0.57,-0.62,-0.83,-1.22,-1.48,-1.89,-2.67,-2.70,-2.60


# Advanced Optimization

Four progressive enhancements to find the best SL / Target combo:

1. **Finer grid** — 0.25% / 0.5% steps for a precise optimum.
2. **Per-setup optimization** — best SL/Target *per timeframe + EMA pair*.
3. **Rank by Total PnL** — favours combos with more decided signals.
4. **Expectancy + Max Drawdown** — risk-adjusted view.

In [146]:
# ── 1. FINER GRID OPTIMIZATION ─────────────────────────────────────────
# SL: 0.5% → 6.0% step 0.25  (23 values)
# TP: 1.0% → 15.0% step 0.5  (29 values)
# 23 * 29 = 667 combinations. Takes a minute or two on 559 signals.

SL_FINE     = [round(x, 2) for x in np.arange(0.5,  6.01, 0.25)]
TARGET_FINE = [round(x, 2) for x in np.arange(1.0, 15.01, 0.5)]

print(f'Sweeping {len(SL_FINE)} x {len(TARGET_FINE)} = {len(SL_FINE)*len(TARGET_FINE)} combinations...')

fine_rows = []
for sl in SL_FINE:
    for tp in TARGET_FINE:
        if tp <= sl:        # skip combos where R:R < 1
            continue
        res = evaluate(signals, stock_data, sl, tp)
        tgt_n = (res['Outcome'] == 'TARGET HIT').sum()
        sl_n  = (res['Outcome'] == 'SL HIT').sum()
        decided = tgt_n + sl_n
        fine_rows.append({
            'SL %':       sl,
            'Target %':   tp,
            'R:R':        round(tp / sl, 2),
            'Total':      len(res),
            'Target Hit': int(tgt_n),
            'SL Hit':     int(sl_n),
            'Win Rate %': round(tgt_n / decided * 100, 1) if decided else 0.0,
            'Avg PnL %':  round(res['PnL %'].mean(), 3),
            'Total PnL %':round(res['PnL %'].sum(),  2),
        })

opt_fine = pd.DataFrame(fine_rows).sort_values('Avg PnL %', ascending=False).reset_index(drop=True)

print('\n' + '=' * 70)
print('  TOP 15 — FINER GRID (ranked by Avg PnL %)')
print('=' * 70)
display(opt_fine.head(15))

best = opt_fine.iloc[0]
print(f"\n  >>> BEST (fine grid) <<<")
print(f"  SL = {best['SL %']}%   Target = {best['Target %']}%   R:R = 1:{best['R:R']}")
print(f"  Win Rate = {best['Win Rate %']}%   Avg PnL = {best['Avg PnL %']}%   Total PnL = {best['Total PnL %']}%")

opt_fine.to_csv(os.path.join(signals_dir, 'optimization_fine_grid.csv'), index=False)
print(f"\nSaved → {os.path.join(signals_dir, 'optimization_fine_grid.csv')}")

Sweeping 23 x 29 = 667 combinations...

  TOP 15 — FINER GRID (ranked by Avg PnL %)


,SL %,Target %,R:R,Total,Target Hit,SL Hit,Win Rate %,Avg PnL %,Total PnL %
0,0.50,2.50,5.00,559,85,470,15.30,0.05,27.96
1,0.50,4.00,8.00,559,63,494,11.30,0.01,5.00
2,0.50,5.50,11.00,559,46,509,8.30,0.01,2.79
3,0.50,5.00,10.00,559,50,506,9.00,-0.00,-1.27
4,0.75,5.50,7.33,559,65,490,11.70,-0.01,-5.71
5,0.50,4.50,9.00,559,54,502,9.70,-0.01,-6.27
6,0.75,4.00,5.33,559,86,471,15.40,-0.02,-9.25
7,0.50,6.00,12.00,559,40,515,7.20,-0.02,-13.21
8,0.75,5.00,6.67,559,69,487,12.40,-0.03,-18.52
9,0.75,6.00,8.00,559,58,497,10.50,-0.04,-20.46



  >>> BEST (fine grid) <<<
  SL = 0.5%   Target = 2.5%   R:R = 1:5.0
  Win Rate = 15.3%   Avg PnL = 0.05%   Total PnL = 27.96%

Saved → signals\optimization_fine_grid.csv


In [147]:
# ── 2. PER-SETUP OPTIMIZATION ──────────────────────────────────────────
# Find the best SL/Target combo SEPARATELY for each (Timeframe, EMA Pair).
# 1d and 1h behave very differently — one global combo is suboptimal.

SL_GRID2     = [round(x, 2) for x in np.arange(1.0, 6.01, 0.5)]
TARGET_GRID2 = [round(x, 2) for x in np.arange(1.0, 12.01, 0.5)]

per_setup_best = []
per_setup_full = []

setups_in_signals = signals[['Timeframe', 'EMA Pair']].drop_duplicates().values.tolist()

for tf, pair in setups_in_signals:
    sub = signals[(signals['Timeframe'] == tf) & (signals['EMA Pair'] == pair)].reset_index(drop=True)
    rows = []
    for sl in SL_GRID2:
        for tp in TARGET_GRID2:
            if tp <= sl:
                continue
            res = evaluate(sub, stock_data, sl, tp)
            tgt_n = (res['Outcome'] == 'TARGET HIT').sum()
            sl_n  = (res['Outcome'] == 'SL HIT').sum()
            decided = tgt_n + sl_n
            rows.append({
                'Timeframe': tf, 'EMA Pair': pair,
                'SL %': sl, 'Target %': tp,
                'R:R': round(tp/sl, 2),
                'Total': len(res),
                'Target Hit': int(tgt_n), 'SL Hit': int(sl_n),
                'Win Rate %': round(tgt_n/decided*100, 1) if decided else 0.0,
                'Avg PnL %': round(res['PnL %'].mean(), 3),
                'Total PnL %': round(res['PnL %'].sum(), 2),
            })
    df_setup = pd.DataFrame(rows).sort_values('Avg PnL %', ascending=False).reset_index(drop=True)
    per_setup_full.append(df_setup)
    per_setup_best.append(df_setup.iloc[0].to_dict())

best_per_setup_df = pd.DataFrame(per_setup_best)

print('=' * 75)
print('  BEST SL / TARGET PER SETUP (max Avg PnL %)')
print('=' * 75)
display(best_per_setup_df)

# Also show top 5 for each setup
for df_setup in per_setup_full:
    tf   = df_setup['Timeframe'].iloc[0]
    pair = df_setup['EMA Pair'].iloc[0]
    print(f'\n── Top 5 for {tf} / {pair} ──')
    display(df_setup.head(5))

pd.concat(per_setup_full).to_csv(os.path.join(signals_dir, 'optimization_per_setup.csv'), index=False)
print(f"\nSaved full per-setup grid → signals/optimization_per_setup.csv")

  BEST SL / TARGET PER SETUP (max Avg PnL %)


,Timeframe,EMA Pair,SL %,Target %,R:R,Total,Target Hit,SL Hit,Win Rate %,Avg PnL %,Total PnL %
0,1d,EMA10/EMA20,1.00,4.00,4.00,229,45,183,19.70,-0.01,-3.00
1,1d,EMA21/EMA50,1.00,6.00,6.00,141,29,112,20.60,0.44,62.00
2,1h,EMA10/EMA20,1.00,1.50,1.50,128,39,88,30.70,-0.22,-27.72
3,1h,EMA21/EMA50,1.00,1.50,1.50,61,26,33,44.10,0.13,7.78



── Top 5 for 1d / EMA10/EMA20 ──


,Timeframe,EMA Pair,SL %,Target %,R:R,Total,Target Hit,SL Hit,Win Rate %,Avg PnL %,Total PnL %
0,1d,EMA10/EMA20,1.00,4.00,4.00,229,45,183,19.70,-0.01,-3.00
1,1d,EMA10/EMA20,1.00,5.00,5.00,229,37,191,16.20,-0.03,-6.00
2,1d,EMA10/EMA20,1.00,4.50,4.50,229,40,188,17.50,-0.04,-8.00
3,1d,EMA10/EMA20,1.00,3.50,3.50,229,48,180,21.10,-0.05,-12.00
4,1d,EMA10/EMA20,1.00,2.50,2.50,229,59,169,25.90,-0.09,-21.50



── Top 5 for 1d / EMA21/EMA50 ──


...


── Top 5 for 1h / EMA10/EMA20 ──


,Timeframe,EMA Pair,SL %,Target %,R:R,Total,Target Hit,SL Hit,Win Rate %,Avg PnL %,Total PnL %
0,1h,EMA10/EMA20,1.00,1.50,1.50,128,39,88,30.70,-0.22,-27.72
1,1h,EMA10/EMA20,1.00,2.00,2.00,128,29,99,22.70,-0.32,-41.00
2,1h,EMA10/EMA20,1.00,5.50,5.50,128,12,115,9.40,-0.37,-47.27
3,1h,EMA10/EMA20,1.00,4.00,4.00,128,16,112,12.50,-0.38,-48.00
4,1h,EMA10/EMA20,1.00,3.00,3.00,128,19,109,14.80,-0.41,-52.00



── Top 5 for 1h / EMA21/EMA50 ──


,Timeframe,EMA Pair,SL %,Target %,R:R,Total,Target Hit,SL Hit,Win Rate %,Avg PnL %,Total PnL %
0,1h,EMA21/EMA50,1.00,1.50,1.50,61,26,33,44.10,0.13,7.78
1,1h,EMA21/EMA50,1.00,2.00,2.00,61,17,43,28.30,-0.15,-9.00
2,1h,EMA21/EMA50,1.00,3.50,3.50,61,9,51,15.00,-0.32,-19.50
3,1h,EMA21/EMA50,1.00,4.00,4.00,61,8,52,13.30,-0.33,-20.00
4,1h,EMA21/EMA50,1.00,3.00,3.00,61,10,50,16.70,-0.33,-20.00



Saved full per-setup grid → signals/optimization_per_setup.csv


In [148]:
# ── 3. RANK BY TOTAL PnL (absolute returns) ────────────────────────────
# Avg PnL ignores how many signals trigger. Total PnL = Avg PnL × N signals,
# so it favours combos that actually fire often (small SL → few signals decided).
# Re-use the fine grid built in step 1.

opt_total = opt_fine.sort_values('Total PnL %', ascending=False).reset_index(drop=True)

print('=' * 70)
print('  TOP 15 — RANKED BY TOTAL PnL %  (absolute return across all signals)')
print('=' * 70)
display(opt_total.head(15))

best_tot = opt_total.iloc[0]
print(f"\n  >>> BEST (by Total PnL) <<<")
print(f"  SL = {best_tot['SL %']}%   Target = {best_tot['Target %']}%   R:R = 1:{best_tot['R:R']}")
print(f"  Win Rate  = {best_tot['Win Rate %']}%")
print(f"  Avg PnL   = {best_tot['Avg PnL %']}%")
print(f"  Total PnL = {best_tot['Total PnL %']}%  across {best_tot['Total']} signals")

# Compare Avg-PnL-best vs Total-PnL-best
best_avg = opt_fine.iloc[0]
print('\n  Avg-PnL winner  : SL={:.2f}%  TP={:.2f}%  Avg={}%  Total={}%'.format(
    best_avg['SL %'], best_avg['Target %'], best_avg['Avg PnL %'], best_avg['Total PnL %']))
print('  Total-PnL winner: SL={:.2f}%  TP={:.2f}%  Avg={}%  Total={}%'.format(
    best_tot['SL %'], best_tot['Target %'], best_tot['Avg PnL %'], best_tot['Total PnL %']))

  TOP 15 — RANKED BY TOTAL PnL %  (absolute return across all signals)


,SL %,Target %,R:R,Total,Target Hit,SL Hit,Win Rate %,Avg PnL %,Total PnL %
0,0.50,2.50,5.00,559,85,470,15.30,0.05,27.96
1,0.50,4.00,8.00,559,63,494,11.30,0.01,5.00
2,0.50,5.50,11.00,559,46,509,8.30,0.01,2.79
3,0.50,5.00,10.00,559,50,506,9.00,-0.00,-1.27
4,0.75,5.50,7.33,559,65,490,11.70,-0.01,-5.71
5,0.50,4.50,9.00,559,54,502,9.70,-0.01,-6.27
6,0.75,4.00,5.33,559,86,471,15.40,-0.02,-9.25
7,0.50,6.00,12.00,559,40,515,7.20,-0.02,-13.21
8,0.75,5.00,6.67,559,69,487,12.40,-0.03,-18.52
9,0.75,6.00,8.00,559,58,497,10.50,-0.04,-20.46



  >>> BEST (by Total PnL) <<<
  SL = 0.5%   Target = 2.5%   R:R = 1:5.0
  Win Rate  = 15.3%
  Avg PnL   = 0.05%
  Total PnL = 27.96%  across 559.0 signals

  Avg-PnL winner  : SL=0.50%  TP=2.50%  Avg=0.05%  Total=27.96%
  Total-PnL winner: SL=0.50%  TP=2.50%  Avg=0.05%  Total=27.96%


In [149]:
# ── 4. EXPECTANCY + MAX DRAWDOWN (risk-adjusted view) ──────────────────
# Expectancy   = win_rate * avg_win + loss_rate * avg_loss   (per-trade $-expectation)
# Max Drawdown = worst peak-to-trough on the cumulative PnL curve (signals in
#                Touch Time order). Tells you the pain you'd have endured.
# Profit Factor= sum(wins) / |sum(losses)|

def metrics_for(sl_pct, tp_pct):
    res = evaluate(signals, stock_data, sl_pct, tp_pct)
    # order by entry time for a realistic equity curve
    res = res.sort_values('Touch Time').reset_index(drop=True)
    pnl = res['PnL %'].astype(float)

    wins   = pnl[pnl > 0]
    losses = pnl[pnl < 0]

    win_rate = len(wins) / len(pnl) if len(pnl) else 0
    avg_win  = wins.mean()  if len(wins)   else 0
    avg_loss = losses.mean() if len(losses) else 0
    expectancy = win_rate * avg_win + (1 - win_rate) * avg_loss

    profit_factor = (wins.sum() / abs(losses.sum())) if losses.sum() != 0 else np.inf

    # equity curve & drawdown (in %-points, additive)
    equity = pnl.cumsum()
    peak   = equity.cummax()
    dd     = equity - peak
    max_dd = dd.min() if len(dd) else 0

    return {
        'SL %':           sl_pct,
        'Target %':       tp_pct,
        'Trades':         len(pnl),
        'Win Rate %':     round(win_rate * 100, 1),
        'Avg Win %':      round(avg_win, 2),
        'Avg Loss %':     round(avg_loss, 2),
        'Expectancy %':   round(expectancy, 3),
        'Profit Factor':  round(profit_factor, 2) if np.isfinite(profit_factor) else 'inf',
        'Total PnL %':    round(pnl.sum(), 2),
        'Max Drawdown %': round(max_dd, 2),
        'Calmar (Tot/|DD|)': round(pnl.sum() / abs(max_dd), 2) if max_dd < 0 else np.inf,
    }

# Coarse-ish grid (full risk metrics for every combo is expensive)
SL_RM     = [round(x, 2) for x in np.arange(1.0, 6.01, 0.5)]
TARGET_RM = [round(x, 2) for x in np.arange(1.5, 12.01, 0.5)]

rm_rows = []
for sl in SL_RM:
    for tp in TARGET_RM:
        if tp <= sl:
            continue
        rm_rows.append(metrics_for(sl, tp))

rm = pd.DataFrame(rm_rows)

print('=' * 90)
print('  TOP 10 BY EXPECTANCY %')
print('=' * 90)
display(rm.sort_values('Expectancy %', ascending=False).head(10).reset_index(drop=True))

print('\n' + '=' * 90)
print('  TOP 10 BY PROFIT FACTOR')
print('=' * 90)
rm_pf = rm[rm['Profit Factor'] != 'inf'].copy()
rm_pf['Profit Factor'] = rm_pf['Profit Factor'].astype(float)
display(rm_pf.sort_values('Profit Factor', ascending=False).head(10).reset_index(drop=True))

print('\n' + '=' * 90)
print('  TOP 10 BY CALMAR  (Total PnL / |Max Drawdown|)  — best risk-adjusted')
print('=' * 90)
rm_cal = rm[np.isfinite(pd.to_numeric(rm['Calmar (Tot/|DD|)'], errors='coerce'))].copy()
display(rm_cal.sort_values('Calmar (Tot/|DD|)', ascending=False).head(10).reset_index(drop=True))

print('\n' + '=' * 90)
print('  SMALLEST DRAWDOWNS (most comfortable to trade)')
print('=' * 90)
display(rm.sort_values('Max Drawdown %', ascending=False).head(10).reset_index(drop=True))

rm.to_csv(os.path.join(signals_dir, 'optimization_risk_adjusted.csv'), index=False)
print(f"\nSaved → signals/optimization_risk_adjusted.csv")

# ── Final consolidated recommendation ──────────────────────────────────
print('\n' + '#' * 70)
print('  FINAL RECOMMENDATIONS')
print('#' * 70)

best_exp  = rm.sort_values('Expectancy %', ascending=False).iloc[0]
best_cal  = rm_cal.sort_values('Calmar (Tot/|DD|)', ascending=False).iloc[0] if len(rm_cal) else best_exp
best_pf   = rm_pf.sort_values('Profit Factor', ascending=False).iloc[0] if len(rm_pf) else best_exp

print(f"\n  Highest Expectancy   : SL={best_exp['SL %']}%  TP={best_exp['Target %']}%  "
      f"Exp={best_exp['Expectancy %']}%  TotPnL={best_exp['Total PnL %']}%  DD={best_exp['Max Drawdown %']}%")
print(f"  Best Risk-Adjusted   : SL={best_cal['SL %']}%  TP={best_cal['Target %']}%  "
      f"Calmar={best_cal['Calmar (Tot/|DD|)']}  TotPnL={best_cal['Total PnL %']}%  DD={best_cal['Max Drawdown %']}%")
print(f"  Best Profit Factor   : SL={best_pf['SL %']}%   TP={best_pf['Target %']}%   "
      f"PF={best_pf['Profit Factor']}  TotPnL={best_pf['Total PnL %']}%  DD={best_pf['Max Drawdown %']}%")

  TOP 10 BY EXPECTANCY %


,SL %,Target %,Trades,Win Rate %,Avg Win %,Avg Loss %,Expectancy %,Profit Factor,Total PnL %,Max Drawdown %,Calmar (Tot/|DD|)
0,1.00,4.00,559,18.80,4.00,-1.00,-0.06,0.93,-32.00,-129.00,-0.25
1,1.00,5.00,559,15.70,4.96,-1.00,-0.06,0.93,-32.27,-146.00,-0.22
2,1.00,5.50,559,14.50,5.42,-1.00,-0.07,0.92,-37.21,-140.00,-0.27
3,1.00,4.50,559,16.80,4.47,-1.00,-0.08,0.91,-42.77,-147.00,-0.29
4,1.00,3.50,559,20.00,3.50,-1.00,-0.10,0.88,-53.00,-138.50,-0.38
5,1.00,6.00,559,12.70,5.89,-1.00,-0.12,0.86,-67.71,-162.00,-0.42
6,1.00,2.00,559,28.60,2.00,-1.00,-0.14,0.81,-77.00,-105.00,-0.73
7,1.00,2.50,559,24.50,2.50,-1.00,-0.14,0.82,-77.50,-129.50,-0.60
8,1.00,3.00,559,21.30,3.00,-1.00,-0.15,0.82,-81.00,-142.00,-0.57
9,1.00,7.50,559,9.70,7.09,-1.00,-0.22,0.76,-119.91,-178.70,-0.67



  TOP 10 BY PROFIT FACTOR


,SL %,Target %,Trades,Win Rate %,Avg Win %,Avg Loss %,Expectancy %,Profit Factor,Total PnL %,Max Drawdown %,Calmar (Tot/|DD|)
0,1.00,4.00,559,18.80,4.00,-1.00,-0.06,0.93,-32.00,-129.00,-0.25
1,1.00,5.00,559,15.70,4.96,-1.00,-0.06,0.93,-32.27,-146.00,-0.22
2,1.00,5.50,559,14.50,5.42,-1.00,-0.07,0.92,-37.21,-140.00,-0.27
3,1.00,4.50,559,16.80,4.47,-1.00,-0.08,0.91,-42.77,-147.00,-0.29
4,1.00,3.50,559,20.00,3.50,-1.00,-0.10,0.88,-53.00,-138.50,-0.38
5,1.00,6.00,559,12.70,5.89,-1.00,-0.12,0.86,-67.71,-162.00,-0.42
6,3.50,4.00,559,42.20,3.98,-3.49,-0.33,0.84,-179.08,-314.00,-0.57
7,1.00,3.00,559,21.30,3.00,-1.00,-0.15,0.82,-81.00,-142.00,-0.57
8,1.00,2.50,559,24.50,2.50,-1.00,-0.14,0.82,-77.50,-129.50,-0.60
9,1.00,2.00,559,28.60,2.00,-1.00,-0.14,0.81,-77.00,-105.00,-0.73



  TOP 10 BY CALMAR  (Total PnL / |Max Drawdown|)  — best risk-adjusted


,SL %,Target %,Trades,Win Rate %,Avg Win %,Avg Loss %,Expectancy %,Profit Factor,Total PnL %,Max Drawdown %,Calmar (Tot/|DD|)
0,1.00,5.00,559,15.70,4.96,-1.00,-0.06,0.93,-32.27,-146.00,-0.22
1,1.00,4.00,559,18.80,4.00,-1.00,-0.06,0.93,-32.00,-129.00,-0.25
2,1.00,5.50,559,14.50,5.42,-1.00,-0.07,0.92,-37.21,-140.00,-0.27
3,1.00,4.50,559,16.80,4.47,-1.00,-0.08,0.91,-42.77,-147.00,-0.29
4,1.00,3.50,559,20.00,3.50,-1.00,-0.10,0.88,-53.00,-138.50,-0.38
5,1.00,6.00,559,12.70,5.89,-1.00,-0.12,0.86,-67.71,-162.00,-0.42
6,1.00,3.00,559,21.30,3.00,-1.00,-0.15,0.82,-81.00,-142.00,-0.57
7,3.50,4.00,559,42.20,3.98,-3.49,-0.33,0.84,-179.08,-314.00,-0.57
8,1.00,2.50,559,24.50,2.50,-1.00,-0.14,0.82,-77.50,-129.50,-0.60
9,1.50,4.50,559,21.10,4.48,-1.50,-0.24,0.80,-129.76,-214.50,-0.60



  SMALLEST DRAWDOWNS (most comfortable to trade)


,SL %,Target %,Trades,Win Rate %,Avg Win %,Avg Loss %,Expectancy %,Profit Factor,Total PnL %,Max Drawdown %,Calmar (Tot/|DD|)
0,1.00,2.00,559,28.60,2.00,-1.00,-0.14,0.81,-77.00,-105.00,-0.73
1,1.00,4.00,559,18.80,4.00,-1.00,-0.06,0.93,-32.00,-129.00,-0.25
2,1.00,2.50,559,24.50,2.50,-1.00,-0.14,0.82,-77.50,-129.50,-0.60
3,1.00,3.50,559,20.00,3.50,-1.00,-0.10,0.88,-53.00,-138.50,-0.38
4,1.00,5.50,559,14.50,5.42,-1.00,-0.07,0.92,-37.21,-140.00,-0.27
5,1.00,3.00,559,21.30,3.00,-1.00,-0.15,0.82,-81.00,-142.00,-0.57
6,1.00,5.00,559,15.70,4.96,-1.00,-0.06,0.93,-32.27,-146.00,-0.22
7,1.00,4.50,559,16.80,4.47,-1.00,-0.08,0.91,-42.77,-147.00,-0.29
8,1.00,6.00,559,12.70,5.89,-1.00,-0.12,0.86,-67.71,-162.00,-0.42
9,1.50,2.00,559,35.60,2.00,-1.50,-0.25,0.74,-138.49,-165.99,-0.83



Saved → signals/optimization_risk_adjusted.csv

######################################################################
  FINAL RECOMMENDATIONS
######################################################################

  Highest Expectancy   : SL=1.0%  TP=4.0%  Exp=-0.061%  TotPnL=-32.0%  DD=-129.0%
  Best Risk-Adjusted   : SL=1.0%  TP=5.0%  Calmar=-0.22  TotPnL=-32.27%  DD=-146.0%
  Best Profit Factor   : SL=1.0%   TP=4.0%   PF=0.93  TotPnL=-32.0%  DD=-129.0%


# Variant: Limit Trades Per Cross Date

Original logic generates **every** touch after a bullish EMA cross as a separate signal — one cross can yield many trades. This section tests two restricted variants:

- **First-1**: only the **1st touch** after each cross is kept.
- **First-2**: only the **first 2 touches** after each cross are kept.

Each variant runs the full optimization pipeline (coarse grid, fine grid, per-setup, total PnL, risk-adjusted) and saves results to `signals/`.

> The cells below do **not** modify any previous work. They build new filtered signal sets and reuse the existing `evaluate()` and `metrics_for()` functions.

In [150]:
# ── BUILD FILTERED SIGNAL SETS ─────────────────────────────────────────
# Group by (Stock, Timeframe, EMA Pair, Cross Time) and keep the first N
# touches ordered by Touch Time.

def first_n_per_cross(sigs, n):
    return (sigs.sort_values(['Stock', 'Timeframe', 'EMA Pair', 'Cross Time', 'Touch Time'])
                .groupby(['Stock', 'Timeframe', 'EMA Pair', 'Cross Time'], as_index=False)
                .head(n)
                .reset_index(drop=True))

signals_first1 = first_n_per_cross(signals, 1)
signals_first2 = first_n_per_cross(signals, 2)

print(f"All touches  : {len(signals)} signals")
print(f"First-1 only : {len(signals_first1)} signals")
print(f"First-2 only : {len(signals_first2)} signals")

# How many crosses had >1 / >2 touches?
per_cross = signals.groupby(['Stock', 'Timeframe', 'EMA Pair', 'Cross Time']).size()
print(f"\nUnique cross events : {len(per_cross)}")
print(f"Crosses with >1 touch: {(per_cross > 1).sum()}")
print(f"Crosses with >2 touch: {(per_cross > 2).sum()}")
print(f"Max touches on one cross: {per_cross.max()}")

# Save the filtered signal sets
signals_first1.to_csv(os.path.join(signals_dir, 'signals_first1_per_cross.csv'), index=False)
signals_first2.to_csv(os.path.join(signals_dir, 'signals_first2_per_cross.csv'), index=False)
print(f"\nSaved → signals/signals_first1_per_cross.csv")
print(f"Saved → signals/signals_first2_per_cross.csv")

All touches  : 559 signals
First-1 only : 128 signals
First-2 only : 231 signals

Unique cross events : 128
Crosses with >1 touch: 103
Crosses with >2 touch: 81
Max touches on one cross: 28

Saved → signals/signals_first1_per_cross.csv
Saved → signals/signals_first2_per_cross.csv


In [151]:
# ── REPORT PIPELINE (reusable) ─────────────────────────────────────────
# One function that produces ALL reports for a given signal set + label.

def run_full_reports(sig_set, label, sl_default=SL_PCT, tp_default=TARGET_PCT):
    tag = label.lower().replace(' ', '_').replace('-', '')
    print('\n' + '#' * 78)
    print(f'  REPORTS FOR: {label}   ({len(sig_set)} signals)')
    print('#' * 78)

    # 1) Baseline outcome summary at default SL/TP
    base = evaluate(sig_set, stock_data, sl_default, tp_default)
    base.to_csv(os.path.join(signals_dir, f'trades_{tag}_sl{sl_default}_tp{tp_default}.csv'), index=False)
    total = len(base)
    tgt = (base['Outcome'] == 'TARGET HIT').sum()
    sl  = (base['Outcome'] == 'SL HIT').sum()
    op  = (base['Outcome'] == 'OPEN').sum()
    decided = tgt + sl
    print(f'\n  Baseline (SL={sl_default}%, TP={tp_default}%):')
    print(f'    Total={total}  TargetHit={tgt}  SLHit={sl}  Open={op}')
    print(f'    Win rate(decided)={tgt/decided*100:.1f}%   Avg PnL={base["PnL %"].mean():.2f}%')

    per_setup = (base.groupby(['Timeframe', 'EMA Pair'])
                 .apply(lambda x: pd.Series({
                     'Total': len(x),
                     'Target Hit': (x['Outcome']=='TARGET HIT').sum(),
                     'SL Hit':     (x['Outcome']=='SL HIT').sum(),
                     'Win Rate %': round((x['Outcome']=='TARGET HIT').mean()*100, 1),
                     'Avg PnL %':  round(x['PnL %'].mean(), 2),
                 })).reset_index())
    print('\n  Per setup:'); display(per_setup)

    # 2) Coarse grid (original 9x9)
    print('\n  -- Coarse grid (top 10 by Avg PnL) --')
    rows = []
    for s in SL_GRID:
        for t in TARGET_GRID:
            r = evaluate(sig_set, stock_data, s, t)
            th = (r['Outcome']=='TARGET HIT').sum(); sh = (r['Outcome']=='SL HIT').sum()
            d = th + sh
            rows.append({'SL %':s, 'Target %':t, 'R:R':round(t/s,2), 'Total':len(r),
                         'Target Hit':int(th), 'SL Hit':int(sh),
                         'Win Rate %': round(th/d*100,1) if d else 0,
                         'Avg PnL %': round(r['PnL %'].mean(),3),
                         'Total PnL %': round(r['PnL %'].sum(),2)})
    coarse = pd.DataFrame(rows).sort_values('Avg PnL %', ascending=False).reset_index(drop=True)
    display(coarse.head(10))
    coarse.to_csv(os.path.join(signals_dir, f'opt_coarse_{tag}.csv'), index=False)

    # 3) Fine grid
    print('\n  -- Fine grid (top 10 by Avg PnL) --')
    rows = []
    for s in SL_FINE:
        for t in TARGET_FINE:
            if t <= s: continue
            r = evaluate(sig_set, stock_data, s, t)
            th = (r['Outcome']=='TARGET HIT').sum(); sh = (r['Outcome']=='SL HIT').sum()
            d = th + sh
            rows.append({'SL %':s, 'Target %':t, 'R:R':round(t/s,2), 'Total':len(r),
                         'Target Hit':int(th), 'SL Hit':int(sh),
                         'Win Rate %': round(th/d*100,1) if d else 0,
                         'Avg PnL %': round(r['PnL %'].mean(),3),
                         'Total PnL %': round(r['PnL %'].sum(),2)})
    fine = pd.DataFrame(rows).sort_values('Avg PnL %', ascending=False).reset_index(drop=True)
    display(fine.head(10))
    fine.to_csv(os.path.join(signals_dir, f'opt_fine_{tag}.csv'), index=False)

    # 4) Total PnL ranking
    print('\n  -- Fine grid (top 10 by Total PnL) --')
    display(fine.sort_values('Total PnL %', ascending=False).head(10).reset_index(drop=True))

    # 5) Per-setup optimization
    print('\n  -- Best SL/TP per setup --')
    ps_best = []
    for tf, pair in sig_set[['Timeframe','EMA Pair']].drop_duplicates().values.tolist():
        sub = sig_set[(sig_set['Timeframe']==tf) & (sig_set['EMA Pair']==pair)].reset_index(drop=True)
        if len(sub)==0: continue
        local = []
        for s in SL_GRID2:
            for t in TARGET_GRID2:
                if t <= s: continue
                r = evaluate(sub, stock_data, s, t)
                th = (r['Outcome']=='TARGET HIT').sum(); sh = (r['Outcome']=='SL HIT').sum()
                d = th + sh
                local.append({'Timeframe':tf,'EMA Pair':pair,'SL %':s,'Target %':t,
                              'R:R':round(t/s,2),'Total':len(r),
                              'Target Hit':int(th),'SL Hit':int(sh),
                              'Win Rate %':round(th/d*100,1) if d else 0,
                              'Avg PnL %':round(r['PnL %'].mean(),3),
                              'Total PnL %':round(r['PnL %'].sum(),2)})
        if local:
            ps_best.append(pd.DataFrame(local).sort_values('Avg PnL %', ascending=False).iloc[0].to_dict())
    ps_best_df = pd.DataFrame(ps_best)
    display(ps_best_df)
    ps_best_df.to_csv(os.path.join(signals_dir, f'opt_per_setup_{tag}.csv'), index=False)

    # 6) Risk-adjusted
    print('\n  -- Risk-adjusted top 10 by Expectancy --')
    # Inline metric calc against THIS sig_set (metrics_for uses global `signals`)
    def _m(s, t):
        r = evaluate(sig_set, stock_data, s, t).sort_values('Touch Time').reset_index(drop=True)
        pnl = r['PnL %'].astype(float)
        w = pnl[pnl>0]; l = pnl[pnl<0]
        wr = len(w)/len(pnl) if len(pnl) else 0
        aw = w.mean() if len(w) else 0
        al = l.mean() if len(l) else 0
        exp = wr*aw + (1-wr)*al
        pf = (w.sum()/abs(l.sum())) if l.sum()!=0 else np.inf
        eq = pnl.cumsum(); dd = (eq - eq.cummax()).min() if len(pnl) else 0
        return {'SL %':s, 'Target %':t, 'Trades':len(pnl),
                'Win Rate %':round(wr*100,1),
                'Avg Win %':round(aw,2), 'Avg Loss %':round(al,2),
                'Expectancy %':round(exp,3),
                'Profit Factor': round(pf,2) if np.isfinite(pf) else 'inf',
                'Total PnL %':round(pnl.sum(),2),
                'Max Drawdown %':round(dd,2),
                'Calmar (Tot/|DD|)': round(pnl.sum()/abs(dd),2) if dd < 0 else np.inf}
    rm_rows = [_m(s,t) for s in SL_RM for t in TARGET_RM if t > s]
    rm_df = pd.DataFrame(rm_rows)
    display(rm_df.sort_values('Expectancy %', ascending=False).head(10).reset_index(drop=True))
    rm_df.to_csv(os.path.join(signals_dir, f'opt_risk_{tag}.csv'), index=False)

    # Final pick
    best_avg = fine.iloc[0]
    best_tot = fine.sort_values('Total PnL %', ascending=False).iloc[0]
    best_exp = rm_df.sort_values('Expectancy %', ascending=False).iloc[0]
    print('\n  >>> SUMMARY PICKS FOR ' + label + ' <<<')
    print(f"  Best Avg PnL   : SL={best_avg['SL %']}%  TP={best_avg['Target %']}%  Avg={best_avg['Avg PnL %']}%  Tot={best_avg['Total PnL %']}%")
    print(f"  Best Total PnL : SL={best_tot['SL %']}%  TP={best_tot['Target %']}%  Avg={best_tot['Avg PnL %']}%  Tot={best_tot['Total PnL %']}%")
    print(f"  Best Expectancy: SL={best_exp['SL %']}%  TP={best_exp['Target %']}%  Exp={best_exp['Expectancy %']}%  DD={best_exp['Max Drawdown %']}%")

    return {'baseline': base, 'coarse': coarse, 'fine': fine,
            'per_setup': ps_best_df, 'risk': rm_df}

print("run_full_reports() ready.")

run_full_reports() ready.


In [152]:
# ── RUN ALL REPORTS: FIRST-1 PER CROSS ─────────────────────────────────
reports_first1 = run_full_reports(signals_first1, 'First1')


##############################################################################
  REPORTS FOR: First1   (128 signals)
##############################################################################

  Baseline (SL=0.7%, TP=3.7%):
    Total=128  TargetHit=24  SLHit=103  Open=1
    Win rate(decided)=18.9%   Avg PnL=0.13%

  Per setup:


,Timeframe,EMA Pair,Total,Target Hit,SL Hit,Win Rate %,Avg PnL %
0,1d,EMA10/EMA20,49.00,10.00,38.00,20.40,0.21
1,1d,EMA21/EMA50,21.00,5.00,16.00,23.80,0.35
2,1h,EMA10/EMA20,42.00,6.00,36.00,14.30,-0.07
3,1h,EMA21/EMA50,16.00,3.00,13.00,18.80,0.12



  -- Coarse grid (top 10 by Avg PnL) --


,SL %,Target %,R:R,Total,Target Hit,SL Hit,Win Rate %,Avg PnL %,Total PnL %
0,1.00,5.00,5.00,128,27,99,21.40,0.29,37.73
1,3.00,5.00,1.67,128,51,73,41.10,0.29,36.80
2,2.50,5.00,2.00,128,45,79,36.30,0.22,28.30
3,1.00,4.00,4.00,128,31,96,24.40,0.22,28.00
4,5.00,4.00,0.80,128,72,50,59.00,0.22,28.02
5,4.00,4.00,1.00,128,66,58,53.20,0.22,27.99
6,1.50,5.00,3.33,128,33,92,26.40,0.22,27.74
7,4.00,5.00,1.25,128,58,65,47.20,0.22,27.72
8,5.00,1.50,0.30,128,100,23,81.30,0.22,27.64
9,1.00,7.00,7.00,128,18,106,14.50,0.20,26.09



  -- Fine grid (top 10 by Avg PnL) --


...


  -- Best SL/TP per setup --


,Timeframe,EMA Pair,SL %,Target %,R:R,Total,Target Hit,SL Hit,Win Rate %,Avg PnL %,Total PnL %
0,1d,EMA10/EMA20,3.50,4.00,1.14,49,30,18,62.50,1.16,57.00
1,1d,EMA21/EMA50,2.50,8.50,3.40,21,10,11,47.60,2.74,57.50
2,1h,EMA10/EMA20,1.00,5.50,5.50,42,5,36,12.20,-0.16,-6.77
3,1h,EMA21/EMA50,1.00,1.50,1.50,16,10,6,62.50,0.56,9.00



  -- Risk-adjusted top 10 by Expectancy --


,SL %,Target %,Trades,Win Rate %,Avg Win %,Avg Loss %,Expectancy %,Profit Factor,Total PnL %,Max Drawdown %,Calmar (Tot/|DD|)
0,3.50,5.00,128,45.30,4.86,-3.46,0.31,1.18,42.80,-92.00,0.47
1,3.50,4.00,128,50.80,3.94,-3.46,0.30,1.19,41.57,-81.00,0.51
2,1.00,5.00,128,21.90,4.88,-1.00,0.29,1.38,37.73,-35.00,1.08
3,1.00,6.50,128,18.00,6.12,-1.00,0.28,1.35,36.79,-40.00,0.92
4,1.00,5.50,128,20.30,5.24,-1.00,0.27,1.35,35.29,-34.00,1.04
5,1.00,8.00,128,15.60,7.10,-1.00,0.27,1.33,35.09,-40.00,0.88
6,1.00,7.50,128,16.40,6.72,-1.00,0.27,1.33,35.09,-40.00,0.88
7,3.00,5.00,128,41.40,4.85,-2.97,0.26,1.17,36.80,-82.00,0.45
8,1.00,6.00,128,18.80,5.68,-1.00,0.25,1.32,33.29,-40.00,0.83
9,1.00,4.50,128,22.70,4.40,-1.00,0.22,1.30,29.73,-37.00,0.80



  >>> SUMMARY PICKS FOR First1 <<<
  Best Avg PnL   : SL=3.5%  TP=5.0%  Avg=0.334%  Tot=42.8%
  Best Total PnL : SL=3.5%  TP=5.0%  Avg=0.334%  Tot=42.8%
  Best Expectancy: SL=3.5%  TP=5.0%  Exp=0.307%  DD=-92.0%


In [153]:
# ── RUN ALL REPORTS: FIRST-2 PER CROSS ─────────────────────────────────
reports_first2 = run_full_reports(signals_first2, 'First2')


##############################################################################
  REPORTS FOR: First2   (231 signals)
##############################################################################

  Baseline (SL=0.7%, TP=3.7%):
    Total=231  TargetHit=40  SLHit=189  Open=2
    Win rate(decided)=17.5%   Avg PnL=0.07%

  Per setup:


,Timeframe,EMA Pair,Total,Target Hit,SL Hit,Win Rate %,Avg PnL %
0,1d,EMA10/EMA20,88.00,15.00,72.00,17.00,0.06
1,1d,EMA21/EMA50,40.00,10.00,30.00,25.00,0.40
2,1h,EMA10/EMA20,71.00,9.00,62.00,12.70,-0.14
3,1h,EMA21/EMA50,32.00,6.00,25.00,18.80,0.15



  -- Coarse grid (top 10 by Avg PnL) --


,SL %,Target %,R:R,Total,Target Hit,SL Hit,Win Rate %,Avg PnL %,Total PnL %
0,1.00,5.00,5.00,231,46,182,20.20,0.21,49.73
1,1.00,4.00,4.00,231,53,176,23.10,0.16,36.00
2,4.00,4.00,1.00,231,117,108,52.00,0.13,30.34
3,4.00,1.50,0.38,231,169,56,75.10,0.10,23.84
4,5.00,1.50,0.30,231,177,47,79.00,0.09,21.49
5,1.50,4.00,2.67,231,66,162,28.90,0.09,20.01
6,1.50,5.00,3.33,231,55,172,24.20,0.08,17.74
7,3.00,4.00,1.33,231,99,127,43.80,0.05,12.42
8,3.00,5.00,1.67,231,86,139,38.20,0.05,12.15
9,1.00,7.00,7.00,231,29,197,12.80,0.05,12.09



  -- Fine grid (top 10 by Avg PnL) --


...


  -- Best SL/TP per setup --


,Timeframe,EMA Pair,SL %,Target %,R:R,Total,Target Hit,SL Hit,Win Rate %,Avg PnL %,Total PnL %
0,1d,EMA10/EMA20,3.50,4.00,1.14,88,52,35,59.80,0.97,85.50
1,1d,EMA21/EMA50,2.50,8.50,3.40,40,14,26,35.00,1.35,54.00
2,1h,EMA10/EMA20,1.00,5.50,5.50,71,10,60,14.30,-0.05,-3.27
3,1h,EMA21/EMA50,1.00,1.50,1.50,32,18,12,60.00,0.52,16.78



  -- Risk-adjusted top 10 by Expectancy --


,SL %,Target %,Trades,Win Rate %,Avg Win %,Avg Loss %,Expectancy %,Profit Factor,Total PnL %,Max Drawdown %,Calmar (Tot/|DD|)
0,1.00,5.50,231,19.00,5.35,-1.00,0.21,1.27,50.29,-58.00,0.87
1,1.00,5.00,231,20.30,4.93,-1.00,0.21,1.27,49.73,-62.00,0.80
2,1.00,6.00,231,17.30,5.81,-1.00,0.18,1.23,43.29,-71.00,0.61
3,3.50,4.00,231,48.90,3.97,-3.46,0.17,1.12,46.42,-141.50,0.33
4,1.00,4.00,231,22.90,4.00,-1.00,0.15,1.20,36.00,-62.00,0.58
5,1.00,6.50,231,15.60,6.26,-1.00,0.13,1.17,32.29,-71.00,0.45
6,1.00,4.50,231,20.80,4.44,-1.00,0.13,1.18,32.23,-66.50,0.48
7,1.00,8.00,231,13.40,7.23,-1.00,0.10,1.13,26.09,-71.00,0.37
8,1.00,7.50,231,13.90,6.82,-1.00,0.08,1.11,21.09,-71.00,0.30
9,1.50,4.00,231,28.60,4.00,-1.50,0.07,1.08,20.01,-90.50,0.22



  >>> SUMMARY PICKS FOR First2 <<<
  Best Avg PnL   : SL=1.0%  TP=5.5%  Avg=0.218%  Tot=50.29%
  Best Total PnL : SL=1.0%  TP=5.5%  Avg=0.218%  Tot=50.29%
  Best Expectancy: SL=1.0%  TP=5.5%  Exp=0.209%  DD=-58.0%


In [154]:
# ── SIDE-BY-SIDE COMPARISON: All vs First-1 vs First-2 ────────────────
# Compare at the default SL_PCT / TARGET_PCT and at each variant's best fine-grid combo.

def quick_stats(sig_set, sl, tp, label):
    r = evaluate(sig_set, stock_data, sl, tp)
    th = (r['Outcome']=='TARGET HIT').sum()
    sh = (r['Outcome']=='SL HIT').sum()
    d  = th + sh
    return {
        'Variant': label,
        'Signals': len(r),
        'SL %': sl, 'Target %': tp,
        'Target Hit': int(th), 'SL Hit': int(sh),
        'Open': int((r['Outcome']=='OPEN').sum()),
        'Win Rate %': round(th/d*100, 1) if d else 0,
        'Avg PnL %':  round(r['PnL %'].mean(), 2),
        'Total PnL %':round(r['PnL %'].sum(), 2),
    }

# Default SL/TP comparison
print('=' * 80)
print(f'  AT DEFAULT SL={SL_PCT}% / TP={TARGET_PCT}%')
print('=' * 80)
default_cmp = pd.DataFrame([
    quick_stats(signals,        SL_PCT, TARGET_PCT, 'All touches'),
    quick_stats(signals_first1, SL_PCT, TARGET_PCT, 'First-1'),
    quick_stats(signals_first2, SL_PCT, TARGET_PCT, 'First-2'),
])
display(default_cmp)

# At each variant's own best Avg-PnL combo (from fine grid)
print('\n' + '=' * 80)
print('  AT EACH VARIANT\'S BEST FINE-GRID COMBO (max Avg PnL)')
print('=' * 80)
best_all_avg = opt_fine.iloc[0]
best_f1_avg  = reports_first1['fine'].iloc[0]
best_f2_avg  = reports_first2['fine'].iloc[0]
best_cmp = pd.DataFrame([
    quick_stats(signals,        float(best_all_avg['SL %']), float(best_all_avg['Target %']), 'All touches'),
    quick_stats(signals_first1, float(best_f1_avg['SL %']),  float(best_f1_avg['Target %']),  'First-1'),
    quick_stats(signals_first2, float(best_f2_avg['SL %']),  float(best_f2_avg['Target %']),  'First-2'),
])
display(best_cmp)

default_cmp.to_csv(os.path.join(signals_dir, 'comparison_default_sltp.csv'), index=False)
best_cmp.to_csv(os.path.join(signals_dir, 'comparison_best_sltp.csv'), index=False)
print('\nSaved → signals/comparison_default_sltp.csv')
print('Saved → signals/comparison_best_sltp.csv')

  AT DEFAULT SL=0.7% / TP=3.7%


,Variant,Signals,SL %,Target %,Target Hit,SL Hit,Open,Win Rate %,Avg PnL %,Total PnL %
0,All touches,559,0.70,3.70,83,474,2,14.90,-0.04,-24.70
1,First-1,128,0.70,3.70,24,103,1,18.90,0.13,16.70
2,First-2,231,0.70,3.70,40,189,2,17.50,0.07,15.70



  AT EACH VARIANT'S BEST FINE-GRID COMBO (max Avg PnL)


,Variant,Signals,SL %,Target %,Target Hit,SL Hit,Open,Win Rate %,Avg PnL %,Total PnL %
0,All touches,559,0.50,2.50,85,470,2,15.30,0.05,27.96
1,First-1,128,3.50,5.00,56,68,4,45.20,0.33,42.80
2,First-2,231,1.00,5.50,42,185,4,18.50,0.22,50.29



Saved → signals/comparison_default_sltp.csv
Saved → signals/comparison_best_sltp.csv


## Master Grid — All Variants × All SL/TP Combos

Combines the three trade-selection variants (**All touches**, **First-1**, **First-2**) with the fine SL/TP grid into a single unified table. Then ranks by Avg PnL, Total PnL, Expectancy, and Calmar to identify the **global champion** configuration.

In [155]:
# ── MASTER GRID: 3 variants × full fine SL/TP grid ─────────────────────
# One row per (Variant, SL, TP) with full risk metrics.
# 3 variants × ~23 × ~29 ≈ 2000 evaluations — takes a few minutes.

def full_metrics(sig_set, sl, tp, variant):
    r = evaluate(sig_set, stock_data, sl, tp).sort_values('Touch Time').reset_index(drop=True)
    pnl = r['PnL %'].astype(float)
    w = pnl[pnl > 0]; l = pnl[pnl < 0]
    tgt_n = (r['Outcome'] == 'TARGET HIT').sum()
    sl_n  = (r['Outcome'] == 'SL HIT').sum()
    op_n  = (r['Outcome'] == 'OPEN').sum()
    d = tgt_n + sl_n
    wr = len(w) / len(pnl) if len(pnl) else 0
    aw = w.mean() if len(w) else 0
    al = l.mean() if len(l) else 0
    exp = wr * aw + (1 - wr) * al
    pf = (w.sum() / abs(l.sum())) if l.sum() != 0 else np.inf
    eq = pnl.cumsum(); dd = (eq - eq.cummax()).min() if len(pnl) else 0
    calmar = (pnl.sum() / abs(dd)) if dd < 0 else np.inf
    return {
        'Variant': variant, 'SL %': sl, 'Target %': tp, 'R:R': round(tp/sl, 2),
        'Signals': len(pnl), 'Target Hit': int(tgt_n), 'SL Hit': int(sl_n), 'Open': int(op_n),
        'Win Rate %':    round(tgt_n/d*100, 1) if d else 0.0,
        'Avg PnL %':     round(pnl.mean(), 3),
        'Total PnL %':   round(pnl.sum(), 2),
        'Avg Win %':     round(aw, 2),
        'Avg Loss %':    round(al, 2),
        'Expectancy %':  round(exp, 3),
        'Profit Factor': round(pf, 2) if np.isfinite(pf) else np.nan,
        'Max Drawdown %':round(dd, 2),
        'Calmar':        round(calmar, 2) if np.isfinite(calmar) else np.nan,
    }

variants_to_test = [
    ('All touches', signals),
    ('First-1',     signals_first1),
    ('First-2',     signals_first2),
]

print(f'Building master grid: 3 variants × {len(SL_FINE)}×{len(TARGET_FINE)} combos (R:R>1 only)...')
master_rows = []
for variant_name, sig_set in variants_to_test:
    for sl in SL_FINE:
        for tp in TARGET_FINE:
            if tp <= sl:
                continue
            master_rows.append(full_metrics(sig_set, sl, tp, variant_name))
    print(f'   ✓ {variant_name} done')

master = pd.DataFrame(master_rows)
master.to_csv(os.path.join(signals_dir, 'master_grid_all_variants.csv'), index=False)
print(f'\nTotal rows in master grid: {len(master)}')
print(f'Saved → signals/master_grid_all_variants.csv')

# ── GLOBAL CHAMPIONS by each metric ─────────────────────────────────────
print('\n' + '#' * 90)
print('  GLOBAL CHAMPIONS  (best across ALL variants and ALL SL/TP combos)')
print('#' * 90)

print('\n[TOP 15 BY AVG PnL %]'); display(master.sort_values('Avg PnL %',   ascending=False).head(15).reset_index(drop=True))
print('\n[TOP 15 BY TOTAL PnL %]'); display(master.sort_values('Total PnL %', ascending=False).head(15).reset_index(drop=True))
print('\n[TOP 15 BY EXPECTANCY %]'); display(master.sort_values('Expectancy %', ascending=False).head(15).reset_index(drop=True))
print('\n[TOP 15 BY PROFIT FACTOR]'); display(master.dropna(subset=['Profit Factor']).sort_values('Profit Factor', ascending=False).head(15).reset_index(drop=True))
print('\n[TOP 15 BY CALMAR (Tot/|DD|)]'); display(master.dropna(subset=['Calmar']).sort_values('Calmar', ascending=False).head(15).reset_index(drop=True))
print('\n[SMALLEST MAX DRAWDOWN]'); display(master.sort_values('Max Drawdown %', ascending=False).head(15).reset_index(drop=True))

# ── ONE-LINE GLOBAL WINNERS ─────────────────────────────────────────────
def champ(df, metric, ascending=False):
    sub = df.dropna(subset=[metric]) if metric in ('Profit Factor', 'Calmar') else df
    return sub.sort_values(metric, ascending=ascending).iloc[0]

print('\n' + '#' * 90)
print('  ONE-LINE WINNERS')
print('#' * 90)
for metric, asc in [('Avg PnL %', False), ('Total PnL %', False),
                    ('Expectancy %', False), ('Profit Factor', False),
                    ('Calmar', False), ('Max Drawdown %', False)]:
    c = champ(master, metric, ascending=asc)
    print(f"  {metric:<18s} : Variant={c['Variant']:<11s}  SL={c['SL %']}%  TP={c['Target %']}%  "
          f"value={c[metric]}  (Signals={int(c['Signals'])}, WinRate={c['Win Rate %']}%, TotPnL={c['Total PnL %']}%)")

# ── BEST PER VARIANT (by each metric) ───────────────────────────────────
print('\n' + '#' * 90)
print('  BEST WITHIN EACH VARIANT')
print('#' * 90)
for variant_name, _ in variants_to_test:
    sub = master[master['Variant'] == variant_name]
    print(f'\n  --- {variant_name} ---')
    for metric in ['Avg PnL %', 'Total PnL %', 'Expectancy %', 'Calmar']:
        s = sub.dropna(subset=[metric]) if metric == 'Calmar' else sub
        b = s.sort_values(metric, ascending=False).iloc[0]
        print(f"    Best {metric:<14s}: SL={b['SL %']}%  TP={b['Target %']}%  "
              f"{metric}={b[metric]}  (TotPnL={b['Total PnL %']}%, WinRate={b['Win Rate %']}%)")

Building master grid: 3 variants × 23×29 combos (R:R>1 only)...
   ✓ All touches done
   ✓ First-1 done
   ✓ First-2 done

Total rows in master grid: 1638
Saved → signals/master_grid_all_variants.csv

##########################################################################################
  GLOBAL CHAMPIONS  (best across ALL variants and ALL SL/TP combos)
##########################################################################################

[TOP 15 BY AVG PnL %]


,Variant,SL %,Target %,R:R,Signals,Target Hit,SL Hit,Open,Win Rate %,Avg PnL %,Total PnL %,Avg Win %,Avg Loss %,Expectancy %,Profit Factor,Max Drawdown %,Calmar
0,First-1,3.50,5.00,1.43,128,56,68,4,45.20,0.33,42.80,4.86,-3.46,0.31,1.18,-92.00,0.47
1,First-1,0.75,7.50,10.00,128,16,109,3,12.80,0.33,42.54,6.90,-0.75,0.33,1.52,-33.75,1.26
2,First-1,0.75,8.00,10.67,128,15,110,3,12.00,0.33,41.79,7.31,-0.75,0.32,1.51,-37.25,1.12
3,First-1,0.75,5.50,7.33,128,21,104,3,16.80,0.33,41.79,5.21,-0.75,0.32,1.54,-28.50,1.47
4,First-1,3.50,4.00,1.14,128,64,61,3,51.20,0.33,41.57,3.94,-3.46,0.30,1.19,-81.00,0.51
5,First-1,0.75,6.50,8.67,128,18,107,3,14.40,0.32,41.04,6.06,-0.75,0.32,1.51,-32.75,1.25
6,First-1,0.75,6.00,8.00,128,19,106,3,15.20,0.30,38.79,5.63,-0.75,0.30,1.49,-33.75,1.15
7,First-1,1.00,5.00,5.00,128,27,99,2,21.40,0.29,37.73,4.88,-1.00,0.29,1.38,-35.00,1.08
8,First-1,3.00,5.00,1.67,128,51,73,4,41.10,0.29,36.80,4.85,-2.97,0.26,1.17,-82.00,0.45
9,First-1,1.00,6.50,6.50,128,21,104,3,16.80,0.29,36.79,6.12,-1.00,0.28,1.35,-40.00,0.92



[TOP 15 BY TOTAL PnL %]


,Variant,SL %,Target %,R:R,Signals,Target Hit,SL Hit,Open,Win Rate %,Avg PnL %,Total PnL %,Avg Win %,Avg Loss %,Expectancy %,Profit Factor,Max Drawdown %,Calmar
0,First-2,1.00,5.50,5.50,231,42,185,4,18.50,0.22,50.29,5.35,-1.00,0.21,1.27,-58.00,0.87
1,First-2,0.75,6.00,8.00,231,32,195,4,14.10,0.22,50.04,5.77,-0.75,0.21,1.34,-61.50,0.81
2,First-2,1.00,5.00,5.00,231,46,182,3,20.20,0.21,49.73,4.93,-1.00,0.21,1.27,-62.00,0.80
3,First-2,0.75,5.50,7.33,231,34,193,4,15.00,0.20,46.54,5.31,-0.75,0.20,1.32,-57.25,0.81
4,First-2,3.50,4.00,1.14,231,112,114,5,49.60,0.20,46.42,3.97,-3.46,0.17,1.12,-141.50,0.33
5,First-2,0.75,6.50,8.67,231,29,198,4,12.80,0.19,44.29,6.22,-0.75,0.18,1.30,-59.50,0.74
6,First-2,1.00,6.00,6.00,231,38,189,4,16.70,0.19,43.29,5.81,-1.00,0.18,1.23,-71.00,0.61
7,First-1,3.50,5.00,1.43,128,56,68,4,45.20,0.33,42.80,4.86,-3.46,0.31,1.18,-92.00,0.47
8,First-1,0.75,7.50,10.00,128,16,109,3,12.80,0.33,42.54,6.90,-0.75,0.33,1.52,-33.75,1.26
9,First-1,0.75,8.00,10.67,128,15,110,3,12.00,0.33,41.79,7.31,-0.75,0.32,1.51,-37.25,1.12



[TOP 15 BY EXPECTANCY %]


...


[TOP 15 BY CALMAR (Tot/|DD|)]


,Variant,SL %,Target %,R:R,Signals,Target Hit,SL Hit,Open,Win Rate %,Avg PnL %,Total PnL %,Avg Win %,Avg Loss %,Expectancy %,Profit Factor,Max Drawdown %,Calmar
0,First-1,0.75,5.50,7.33,128,21,104,3,16.80,0.33,41.79,5.21,-0.75,0.32,1.54,-28.50,1.47
1,First-1,0.75,7.50,10.00,128,16,109,3,12.80,0.33,42.54,6.90,-0.75,0.33,1.52,-33.75,1.26
2,First-1,0.75,6.50,8.67,128,18,107,3,14.40,0.32,41.04,6.06,-0.75,0.32,1.51,-32.75,1.25
3,First-1,0.75,6.00,8.00,128,19,106,3,15.20,0.30,38.79,5.63,-0.75,0.30,1.49,-33.75,1.15
4,First-1,0.75,8.00,10.67,128,15,110,3,12.00,0.33,41.79,7.31,-0.75,0.32,1.51,-37.25,1.12
5,First-1,0.75,5.00,6.67,128,22,104,2,17.50,0.26,33.73,4.86,-0.75,0.26,1.43,-30.00,1.12
6,First-1,1.00,5.00,5.00,128,27,99,2,21.40,0.29,37.73,4.88,-1.00,0.29,1.38,-35.00,1.08
7,First-1,1.00,5.50,5.50,128,24,101,3,19.20,0.28,35.29,5.24,-1.00,0.27,1.35,-34.00,1.04
8,First-1,0.75,7.00,9.33,128,16,109,3,12.80,0.27,34.54,6.46,-0.75,0.26,1.42,-34.75,0.99
9,First-1,0.50,6.50,13.00,128,13,112,3,10.40,0.26,32.79,5.92,-0.50,0.25,1.59,-35.00,0.94



[SMALLEST MAX DRAWDOWN]


,Variant,SL %,Target %,R:R,Signals,Target Hit,SL Hit,Open,Win Rate %,Avg PnL %,Total PnL %,Avg Win %,Avg Loss %,Expectancy %,Profit Factor,Max Drawdown %,Calmar
0,First-1,0.50,2.00,4.00,128,25,102,1,19.70,-0.01,-1.00,2.00,-0.50,-0.01,0.98,-22.50,-0.04
1,First-1,0.75,2.00,2.67,128,34,93,1,26.80,-0.01,-1.75,2.00,-0.75,-0.02,0.97,-25.00,-0.07
2,First-1,1.50,2.00,1.33,128,56,70,2,44.40,0.05,6.01,2.00,-1.49,0.04,1.06,-28.00,0.21
3,First-1,1.25,2.00,1.60,128,48,79,1,37.80,-0.02,-2.75,2.00,-1.25,-0.03,0.97,-28.50,-0.10
4,First-1,0.75,5.50,7.33,128,21,104,3,16.80,0.33,41.79,5.21,-0.75,0.32,1.54,-28.50,1.47
5,First-2,0.50,2.00,4.00,231,46,182,2,20.20,0.01,3.30,2.01,-0.50,0.01,1.04,-28.70,0.11
6,First-1,0.75,5.00,6.67,128,22,104,2,17.50,0.26,33.73,4.86,-0.75,0.26,1.43,-30.00,1.12
7,First-1,1.00,2.00,2.00,128,41,86,1,32.30,-0.03,-4.00,2.00,-1.00,-0.04,0.95,-31.00,-0.13
8,First-1,1.75,2.00,1.14,128,58,67,3,46.40,-0.02,-2.18,1.97,-1.74,-0.03,0.98,-31.25,-0.07
9,First-1,0.75,4.50,6.00,128,23,103,2,18.30,0.22,27.98,4.38,-0.75,0.21,1.36,-31.50,0.89



##########################################################################################
  ONE-LINE WINNERS
##########################################################################################
  Avg PnL %          : Variant=First-1      SL=3.5%  TP=5.0%  value=0.334  (Signals=128, WinRate=45.2%, TotPnL=42.8%)
  Total PnL %        : Variant=First-2      SL=1.0%  TP=5.5%  value=50.29  (Signals=231, WinRate=18.5%, TotPnL=50.29%)
  Expectancy %       : Variant=First-1      SL=0.75%  TP=7.5%  value=0.326  (Signals=128, WinRate=12.8%, TotPnL=42.54%)
  Profit Factor      : Variant=First-1      SL=0.5%  TP=6.5%  value=1.59  (Signals=128, WinRate=10.4%, TotPnL=32.79%)
  Calmar             : Variant=First-1      SL=0.75%  TP=5.5%  value=1.47  (Signals=128, WinRate=16.8%, TotPnL=41.79%)
  Max Drawdown %     : Variant=First-1      SL=0.5%  TP=2.0%  value=-22.5  (Signals=128, WinRate=19.7%, TotPnL=-1.0%)

#####################################################################################

# Per-EMA-Pair Optimization — 4 Reports Each

Each (Timeframe + EMA Pair) setup is optimized **independently** — the optimum SL/Target for `1d EMA21/EMA50` will likely differ from `1h EMA10/EMA20`. For every setup, the section below produces **4 reports**:

| # | Report | What it tells you |
|---|--------|-------------------|
| 1 | Top 15 by **Avg PnL %** | Best per-trade quality combo |
| 2 | Top 15 by **Total PnL %** | Best absolute return combo |
| 3 | Top 15 by **Expectancy / Risk-adjusted** | Best risk-adjusted (with Profit Factor, Drawdown, Calmar) |
| 4 | **Per-Stock breakdown** at the best combo | Which stocks drive the result |

So **4 setups × 4 reports = 16 reports** total. All CSVs saved with `setup_<tf>_<pair>` tags.

In [156]:
# ── PER-EMA-PAIR OPTIMIZATION (4 reports per setup) ────────────────────
# Configurable source: change to signals_first1 or signals_first2 to study those variants.
SOURCE_SIGNALS = signals
SOURCE_LABEL   = 'all_touches'        # 'first1' or 'first2' if you swap above

# Grid used for per-setup sweeps (a touch coarser than master grid to keep it fast)
SL_PAIR     = [round(x, 2) for x in np.arange(0.5, 6.01, 0.25)]
TARGET_PAIR = [round(x, 2) for x in np.arange(1.0, 15.01, 0.5)]

def evaluate_one_combo(sig_set, sl, tp):
    """Return (per-trade DataFrame, summary dict) for a single SL/TP."""
    r = evaluate(sig_set, stock_data, sl, tp).sort_values('Touch Time').reset_index(drop=True)
    pnl = r['PnL %'].astype(float)
    w = pnl[pnl > 0]; l = pnl[pnl < 0]
    tgt_n = (r['Outcome'] == 'TARGET HIT').sum()
    sl_n  = (r['Outcome'] == 'SL HIT').sum()
    op_n  = (r['Outcome'] == 'OPEN').sum()
    d = tgt_n + sl_n
    wr = len(w) / len(pnl) if len(pnl) else 0
    aw = w.mean() if len(w) else 0
    al = l.mean() if len(l) else 0
    exp = wr * aw + (1 - wr) * al
    pf = (w.sum() / abs(l.sum())) if l.sum() != 0 else np.inf
    eq = pnl.cumsum(); dd = (eq - eq.cummax()).min() if len(pnl) else 0
    calmar = (pnl.sum() / abs(dd)) if dd < 0 else np.inf
    summary = {
        'SL %': sl, 'Target %': tp, 'R:R': round(tp/sl, 2),
        'Signals': len(pnl), 'Target Hit': int(tgt_n), 'SL Hit': int(sl_n), 'Open': int(op_n),
        'Win Rate %':    round(tgt_n/d*100, 1) if d else 0.0,
        'Avg PnL %':     round(pnl.mean(), 3),
        'Total PnL %':   round(pnl.sum(), 2),
        'Avg Win %':     round(aw, 2),
        'Avg Loss %':    round(al, 2),
        'Expectancy %':  round(exp, 3),
        'Profit Factor': round(pf, 2) if np.isfinite(pf) else np.nan,
        'Max Drawdown %':round(dd, 2),
        'Calmar':        round(calmar, 2) if np.isfinite(calmar) else np.nan,
    }
    return r, summary

setups_in_signals = SOURCE_SIGNALS[['Timeframe', 'EMA Pair']].drop_duplicates().values.tolist()
all_setup_grids = {}
overall_winners = []

for tf, pair in setups_in_signals:
    sub = SOURCE_SIGNALS[(SOURCE_SIGNALS['Timeframe'] == tf) &
                         (SOURCE_SIGNALS['EMA Pair'] == pair)].reset_index(drop=True)
    setup_tag = f"{tf}_{pair.replace('/', '_')}"
    setup_label = f"{tf}  {pair}"

    print('\n' + '█' * 92)
    print(f'  SETUP: {setup_label}    ({len(sub)} signals)')
    print('█' * 92)

    if len(sub) == 0:
        print('  No signals for this setup. Skipping.')
        continue

    # Build grid of summaries
    summaries = []
    for s in SL_PAIR:
        for t in TARGET_PAIR:
            if t <= s:
                continue
            _, summ = evaluate_one_combo(sub, s, t)
            summaries.append(summ)
    grid = pd.DataFrame(summaries)
    grid.to_csv(os.path.join(signals_dir, f'pair_opt_grid_{SOURCE_LABEL}_{setup_tag}.csv'), index=False)
    all_setup_grids[(tf, pair)] = grid

    # ── REPORT 1: top by Avg PnL ─────────────────────────────────────────
    print('\n  📊 REPORT 1 — Top 15 by Avg PnL %')
    r1 = grid.sort_values('Avg PnL %', ascending=False).head(15).reset_index(drop=True)
    display(r1)
    best_avg = grid.sort_values('Avg PnL %', ascending=False).iloc[0]
    r1.to_csv(os.path.join(signals_dir, f'pair_top_avg_{SOURCE_LABEL}_{setup_tag}.csv'), index=False)

    # ── REPORT 2: top by Total PnL ───────────────────────────────────────
    print('\n  📈 REPORT 2 — Top 15 by Total PnL %')
    r2 = grid.sort_values('Total PnL %', ascending=False).head(15).reset_index(drop=True)
    display(r2)
    best_tot = grid.sort_values('Total PnL %', ascending=False).iloc[0]
    r2.to_csv(os.path.join(signals_dir, f'pair_top_total_{SOURCE_LABEL}_{setup_tag}.csv'), index=False)

    # ── REPORT 3: risk-adjusted (top by Expectancy, with PF/DD/Calmar visible) ─
    print('\n  ⚖️  REPORT 3 — Top 15 by Expectancy (risk-adjusted view)')
    r3 = grid.sort_values('Expectancy %', ascending=False).head(15).reset_index(drop=True)
    display(r3)
    best_exp = grid.sort_values('Expectancy %', ascending=False).iloc[0]
    best_cal = grid.dropna(subset=['Calmar']).sort_values('Calmar', ascending=False).iloc[0] if grid['Calmar'].notna().any() else best_exp
    best_pf  = grid.dropna(subset=['Profit Factor']).sort_values('Profit Factor', ascending=False).iloc[0] if grid['Profit Factor'].notna().any() else best_exp
    r3.to_csv(os.path.join(signals_dir, f'pair_top_risk_{SOURCE_LABEL}_{setup_tag}.csv'), index=False)

    # ── REPORT 4: per-stock breakdown at this setup's best Avg-PnL combo ─
    print(f"\n  🏢 REPORT 4 — Per-stock breakdown at best Avg-PnL combo "
          f"(SL={best_avg['SL %']}% / TP={best_avg['Target %']}%)")
    trades_best, _ = evaluate_one_combo(sub, float(best_avg['SL %']), float(best_avg['Target %']))
    trades_best.to_csv(os.path.join(signals_dir, f'pair_trades_best_{SOURCE_LABEL}_{setup_tag}.csv'), index=False)
    per_stock = (trades_best.groupby('Stock')
                 .apply(lambda x: pd.Series({
                     'Trades':     len(x),
                     'Target Hit': (x['Outcome']=='TARGET HIT').sum(),
                     'SL Hit':     (x['Outcome']=='SL HIT').sum(),
                     'Open':       (x['Outcome']=='OPEN').sum(),
                     'Win Rate %': round((x['Outcome']=='TARGET HIT').mean()*100, 1),
                     'Avg PnL %':  round(x['PnL %'].mean(), 2),
                     'Total PnL %':round(x['PnL %'].sum(), 2),
                 })).reset_index())
    display(per_stock)
    per_stock.to_csv(os.path.join(signals_dir, f'pair_per_stock_best_{SOURCE_LABEL}_{setup_tag}.csv'), index=False)

    # ── one-line winners for this setup ──────────────────────────────────
    print(f"\n  🏆 WINNERS FOR {setup_label}:")
    print(f"     Best Avg PnL    : SL={best_avg['SL %']}%  TP={best_avg['Target %']}%  "
          f"Avg={best_avg['Avg PnL %']}%  Tot={best_avg['Total PnL %']}%  WR={best_avg['Win Rate %']}%")
    print(f"     Best Total PnL  : SL={best_tot['SL %']}%  TP={best_tot['Target %']}%  "
          f"Avg={best_tot['Avg PnL %']}%  Tot={best_tot['Total PnL %']}%  WR={best_tot['Win Rate %']}%")
    print(f"     Best Expectancy : SL={best_exp['SL %']}%  TP={best_exp['Target %']}%  "
          f"Exp={best_exp['Expectancy %']}%  DD={best_exp['Max Drawdown %']}%")
    print(f"     Best Profit Fct : SL={best_pf['SL %']}%   TP={best_pf['Target %']}%   "
          f"PF={best_pf['Profit Factor']}  Tot={best_pf['Total PnL %']}%")
    print(f"     Best Calmar     : SL={best_cal['SL %']}%  TP={best_cal['Target %']}%  "
          f"Calmar={best_cal['Calmar']}  DD={best_cal['Max Drawdown %']}%")

    overall_winners.append({
        'Timeframe': tf, 'EMA Pair': pair, 'Signals': len(sub),
        'Best Avg SL %':      best_avg['SL %'],
        'Best Avg TP %':      best_avg['Target %'],
        'Best Avg PnL %':     best_avg['Avg PnL %'],
        'Best Total SL %':    best_tot['SL %'],
        'Best Total TP %':    best_tot['Target %'],
        'Best Total PnL %':   best_tot['Total PnL %'],
        'Best Exp SL %':      best_exp['SL %'],
        'Best Exp TP %':      best_exp['Target %'],
        'Best Expectancy %':  best_exp['Expectancy %'],
        'Best Calmar SL %':   best_cal['SL %'],
        'Best Calmar TP %':   best_cal['Target %'],
        'Best Calmar':        best_cal['Calmar'],
    })

# ── CONSOLIDATED WINNERS TABLE ──────────────────────────────────────────
print('\n' + '═' * 92)
print('  CONSOLIDATED WINNERS — best SL/TP per EMA setup, by each metric')
print('═' * 92)
winners_df = pd.DataFrame(overall_winners)
display(winners_df)
winners_df.to_csv(os.path.join(signals_dir, f'pair_winners_consolidated_{SOURCE_LABEL}.csv'), index=False)
print(f'\nSaved → signals/pair_winners_consolidated_{SOURCE_LABEL}.csv')


████████████████████████████████████████████████████████████████████████████████████████████
  SETUP: 1d  EMA10/EMA20    (229 signals)
████████████████████████████████████████████████████████████████████████████████████████████

  📊 REPORT 1 — Top 15 by Avg PnL %


,SL %,Target %,R:R,Signals,Target Hit,SL Hit,Open,Win Rate %,Avg PnL %,Total PnL %,Avg Win %,Avg Loss %,Expectancy %,Profit Factor,Max Drawdown %,Calmar
0,0.50,2.50,5.00,229,36,191,1,15.90,0.09,19.73,3.11,-0.50,0.08,1.21,-15.50,1.27
1,0.50,4.00,8.00,229,25,203,1,11.00,-0.01,-1.50,4.00,-0.50,-0.01,0.99,-24.00,-0.06
2,1.00,4.00,4.00,229,45,183,1,19.70,-0.01,-3.00,4.00,-1.00,-0.02,0.98,-39.00,-0.08
3,0.50,5.00,10.00,229,20,208,1,8.80,-0.02,-4.00,5.00,-0.50,-0.02,0.96,-27.00,-0.15
4,0.50,4.50,9.00,229,22,206,1,9.60,-0.02,-4.00,4.50,-0.50,-0.02,0.96,-30.00,-0.13
5,1.00,5.00,5.00,229,37,191,1,16.20,-0.03,-6.00,5.00,-1.00,-0.03,0.97,-42.00,-0.14
6,1.00,4.50,4.50,229,40,188,1,17.50,-0.04,-8.00,4.50,-1.00,-0.04,0.96,-43.00,-0.19
7,0.75,4.00,5.33,229,34,194,1,14.90,-0.04,-9.50,4.00,-0.75,-0.04,0.93,-33.00,-0.29
8,0.50,2.00,4.00,229,41,187,1,18.00,-0.05,-11.50,2.00,-0.50,-0.05,0.88,-22.50,-0.51
9,1.00,3.50,3.50,229,48,180,1,21.10,-0.05,-12.00,3.50,-1.00,-0.06,0.93,-43.50,-0.28



  📈 REPORT 2 — Top 15 by Total PnL %


,SL %,Target %,R:R,Signals,Target Hit,SL Hit,Open,Win Rate %,Avg PnL %,Total PnL %,Avg Win %,Avg Loss %,Expectancy %,Profit Factor,Max Drawdown %,Calmar
0,0.50,2.50,5.00,229,36,191,1,15.90,0.09,19.73,3.11,-0.50,0.08,1.21,-15.50,1.27
1,0.50,4.00,8.00,229,25,203,1,11.00,-0.01,-1.50,4.00,-0.50,-0.01,0.99,-24.00,-0.06
2,1.00,4.00,4.00,229,45,183,1,19.70,-0.01,-3.00,4.00,-1.00,-0.02,0.98,-39.00,-0.08
3,0.50,5.00,10.00,229,20,208,1,8.80,-0.02,-4.00,5.00,-0.50,-0.02,0.96,-27.00,-0.15
4,0.50,4.50,9.00,229,22,206,1,9.60,-0.02,-4.00,4.50,-0.50,-0.02,0.96,-30.00,-0.13
5,1.00,5.00,5.00,229,37,191,1,16.20,-0.03,-6.00,5.00,-1.00,-0.03,0.97,-42.00,-0.14
6,1.00,4.50,4.50,229,40,188,1,17.50,-0.04,-8.00,4.50,-1.00,-0.04,0.96,-43.00,-0.19
7,0.75,4.00,5.33,229,34,194,1,14.90,-0.04,-9.50,4.00,-0.75,-0.04,0.93,-33.00,-0.29
8,0.50,2.00,4.00,229,41,187,1,18.00,-0.05,-11.50,2.00,-0.50,-0.05,0.88,-22.50,-0.51
9,1.00,3.50,3.50,229,48,180,1,21.10,-0.05,-12.00,3.50,-1.00,-0.06,0.93,-43.50,-0.28



  ⚖️  REPORT 3 — Top 15 by Expectancy (risk-adjusted view)


...


  🏢 REPORT 4 — Per-stock breakdown at best Avg-PnL combo (SL=0.5% / TP=1.5%)


,Stock,Trades,Target Hit,SL Hit,Open,Win Rate %,Avg PnL %,Total PnL %
0,HDFCBANK,5.00,1.00,4.00,0.00,20.00,-0.10,-0.50
1,ICICIBANK,13.00,5.00,8.00,0.00,38.50,0.27,3.50
2,INFY,17.00,5.00,12.00,0.00,29.40,0.09,1.50
3,RELIANCE,17.00,5.00,10.00,1.00,29.40,0.25,4.28
4,TCS,9.00,4.00,5.00,0.00,44.40,0.39,3.50



  🏆 WINNERS FOR 1h  EMA21/EMA50:
     Best Avg PnL    : SL=0.5%  TP=1.5%  Avg=0.201%  Tot=12.28%  WR=33.9%
     Best Total PnL  : SL=0.5%  TP=1.5%  Avg=0.201%  Tot=12.28%  WR=33.9%
     Best Expectancy : SL=0.5%  TP=1.5%  Exp=0.193%  DD=-3.0%
     Best Profit Fct : SL=0.5%   TP=1.5%   PF=1.63  Tot=12.28%
     Best Calmar     : SL=0.5%  TP=1.5%  Calmar=4.09  DD=-3.0%

════════════════════════════════════════════════════════════════════════════════════════════
  CONSOLIDATED WINNERS — best SL/TP per EMA setup, by each metric
════════════════════════════════════════════════════════════════════════════════════════════


,Timeframe,EMA Pair,Signals,Best Avg SL %,Best Avg TP %,Best Avg PnL %,Best Total SL %,Best Total TP %,Best Total PnL %,Best Exp SL %,Best Exp TP %,Best Expectancy %,Best Calmar SL %,Best Calmar TP %,Best Calmar
0,1d,EMA10/EMA20,229,0.50,2.50,0.09,0.50,2.50,19.73,0.50,2.50,0.08,0.50,2.50,1.27
1,1d,EMA21/EMA50,141,0.75,6.00,0.45,0.75,6.00,63.00,0.75,6.00,0.45,0.50,2.50,2.08
2,1h,EMA10/EMA20,128,0.50,1.50,-0.01,0.50,1.50,-1.28,0.50,1.50,-0.01,0.50,1.50,-0.11
3,1h,EMA21/EMA50,61,0.50,1.50,0.20,0.50,1.50,12.28,0.50,1.50,0.19,0.50,1.50,4.09



Saved → signals/pair_winners_consolidated_all_touches.csv
